In [ ]:
%%writefile bfs.cpp
#include <iostream>
#include <vector>
#include <queue>
#include <chrono>

using namespace std;
using namespace chrono;

void bfs(int V, vector<int>& row_ptr, vector<int>& col_idx, int source, vector<int>& level) {
    fill(level.begin(), level.end(), -1);
    level[source] = 0;
    queue<int> q;
    q.push(source);
    while (!q.empty()) {
        int u = q.front(); q.pop();
        for (int e = row_ptr[u]; e < row_ptr[u + 1]; e++) {
            int v = col_idx[e];
            if (level[v] == -1) {
                level[v] = level[u] + 1;
                q.push(v);
            }
        }
    }
}

void generate_graph(int V, int avg_degree, vector<int>& row_ptr, vector<int>& col_idx) {
    srand(42);
    vector<vector<int>> adj(V);
    for (int u = 0; u < V; u++) {
        for (int i = 0; i < avg_degree; i++) {
            int v = rand() % V;
            if (v != u) adj[u].push_back(v);
        }
    }
    row_ptr.resize(V + 1, 0);
    for (int u = 0; u < V; u++)
        row_ptr[u + 1] = row_ptr[u] + adj[u].size();
    for (int u = 0; u < V; u++)
        for (int v : adj[u])
            col_idx.push_back(v);
}

void run_test(const string& label, int V, int avg_degree) {
    vector<int> row_ptr, col_idx;
    generate_graph(V, avg_degree, row_ptr, col_idx);
    int E = col_idx.size();
    vector<int> level(V);

    size_t mem_bytes = (size_t)(V + 1) * sizeof(int)   // row_ptr
                     + (size_t)E       * sizeof(int)    // col_idx
                     + (size_t)V       * sizeof(int)    // level
                     + (size_t)V       * sizeof(int);   // queue worst case
    double mem_mb = mem_bytes / (1024.0 * 1024.0);

    auto start = high_resolution_clock::now();
    bfs(V, row_ptr, col_idx, 0, level);
    auto end = high_resolution_clock::now();

    double ms = duration_cast<nanoseconds>(end - start).count() / 1e6;

    int visited = 0;
    for (int l : level) if (l != -1) visited++;

    cout << "=============================" << endl;
    cout << "  Dataset     : " << label     << endl;
    cout << "  Nodes       : " << V         << endl;
    cout << "  Edges       : " << E         << endl;
    cout << "  Visited     : " << visited   << endl;
    cout << "  Time        : " << ms        << " ms" << endl;
    cout << "  CPU Mem     : " << mem_mb    << " MB" << endl;
    cout << "  GPU Mem     : 0.00 MB (CPU only)" << endl;
    cout << "  Total Mem   : " << mem_mb    << " MB" << endl;
    cout << "  Throughput  : " << (int)(E / ms) << " edges/ms" << endl;
    cout << "=============================" << endl << endl;
}

int main() {
    run_test("Small",   1000,    8);
    run_test("Medium",  64000,   8);
    run_test("Large",   256000,  8);
    run_test("XLarge",  1000000, 8);
    return 0;
}

Writing bfs.cpp


In [ ]:
!g++ -o bfs bfs.cpp

In [ ]:
!./bfs

  Dataset     : Small
  Nodes       : 1000
  Edges       : 7992
  Visited     : 999
  Time        : 0.186962 ms
  CPU Mem     : 0.041935 MB
  GPU Mem     : 0.00 MB (CPU only)
  Total Mem   : 0.041935 MB
  Throughput  : 42746 edges/ms

  Dataset     : Medium
  Nodes       : 64000
  Edges       : 511993
  Visited     : 63985
  Time        : 20.6036 ms
  CPU Mem     : 2.68552 MB
  GPU Mem     : 0.00 MB (CPU only)
  Total Mem   : 2.68552 MB
  Throughput  : 24849 edges/ms

  Dataset     : Large
  Nodes       : 256000
  Edges       : 2047993
  Visited     : 255910
  Time        : 117.106 ms
  CPU Mem     : 10.7422 MB
  GPU Mem     : 0.00 MB (CPU only)
  Total Mem   : 10.7422 MB
  Throughput  : 17488 edges/ms

  Dataset     : XLarge
  Nodes       : 1000000
  Edges       : 7999994
  Visited     : 999652
  Time        : 823.863 ms
  CPU Mem     : 41.9617 MB
  GPU Mem     : 0.00 MB (CPU only)
  Total Mem   : 41.9617 MB
  Throughput  : 9710 edges/ms



In [ ]:
%%writefile bfs_cuda.cu
#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

#define THREADS_PER_BLOCK 256
#define INF -1

__global__ void bfs_kernel(
    const int* __restrict__ row_ptr,
    const int* __restrict__ col_idx,
    int* level,
    const int* frontier,
    int* next_frontier,
    int* next_size,
    int frontier_size,
    int current_level)
{
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid >= frontier_size) return;
    int u = frontier[tid];
    for (int e = row_ptr[u]; e < row_ptr[u + 1]; e++) {
        int v = col_idx[e];
        if (atomicCAS(&level[v], INF, current_level + 1) == INF) {
            int pos = atomicAdd(next_size, 1);
            next_frontier[pos] = v;
        }
    }
}

void bfs_cuda(int V, int E, const int* h_row_ptr, const int* h_col_idx, int source, int* h_level) {
    for (int i = 0; i < V; i++) h_level[i] = INF;
    h_level[source] = 0;

    int *d_row_ptr, *d_col_idx, *d_level;
    int *d_frontier, *d_next_frontier, *d_next_size;

    cudaMalloc(&d_row_ptr,       (V+1)*sizeof(int));
    cudaMalloc(&d_col_idx,       E*sizeof(int));
    cudaMalloc(&d_level,         V*sizeof(int));
    cudaMalloc(&d_frontier,      V*sizeof(int));
    cudaMalloc(&d_next_frontier, V*sizeof(int));
    cudaMalloc(&d_next_size,     sizeof(int));

    cudaMemcpy(d_row_ptr, h_row_ptr, (V+1)*sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_col_idx, h_col_idx, E*sizeof(int),     cudaMemcpyHostToDevice);
    cudaMemcpy(d_level,   h_level,   V*sizeof(int),     cudaMemcpyHostToDevice);

    int h_frontier_size = 1;
    cudaMemcpy(d_frontier, &source, sizeof(int), cudaMemcpyHostToDevice);

    for (int lvl = 0; h_frontier_size > 0; lvl++) {
        int zero = 0;
        cudaMemcpy(d_next_size, &zero, sizeof(int), cudaMemcpyHostToDevice);
        int blocks = (h_frontier_size + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;
        bfs_kernel<<<blocks, THREADS_PER_BLOCK>>>(
            d_row_ptr, d_col_idx, d_level,
            d_frontier, d_next_frontier, d_next_size,
            h_frontier_size, lvl);
        cudaDeviceSynchronize();
        cudaMemcpy(&h_frontier_size, d_next_size, sizeof(int), cudaMemcpyDeviceToHost);
        int* tmp = d_frontier;
        d_frontier = d_next_frontier;
        d_next_frontier = tmp;
    }

    cudaMemcpy(h_level, d_level, V*sizeof(int), cudaMemcpyDeviceToHost);

    cudaFree(d_row_ptr); cudaFree(d_col_idx); cudaFree(d_level);
    cudaFree(d_frontier); cudaFree(d_next_frontier); cudaFree(d_next_size);
}

void generate_graph(int V, int avg_degree, int** out_row_ptr, int** out_col_idx, int* out_E) {
    srand(42);
    int** adj    = (int**)calloc(V, sizeof(int*));
    int*  adj_sz = (int*)calloc(V, sizeof(int));
    for (int u = 0; u < V; u++) {
        adj[u] = (int*)malloc(avg_degree * 2 * sizeof(int));
        for (int i = 0; i < avg_degree; i++) {
            int v = rand() % V;
            if (v != u) adj[u][adj_sz[u]++] = v;
        }
    }
    int* row_ptr = (int*)calloc(V + 1, sizeof(int));
    for (int u = 0; u < V; u++) row_ptr[u+1] = row_ptr[u] + adj_sz[u];
    int E = row_ptr[V];
    int* col_idx = (int*)malloc(E * sizeof(int));
    for (int u = 0; u < V; u++)
        for (int i = 0; i < adj_sz[u]; i++)
            col_idx[row_ptr[u] + i] = adj[u][i];
    for (int u = 0; u < V; u++) free(adj[u]);
    free(adj); free(adj_sz);
    *out_row_ptr = row_ptr;
    *out_col_idx = col_idx;
    *out_E = E;
}

void run_test(const char* label, int V, int avg_degree) {
    int *row_ptr, *col_idx, E;
    generate_graph(V, avg_degree, &row_ptr, &col_idx, &E);
    int* h_level = (int*)malloc(V * sizeof(int));

    size_t gpu_bytes = (size_t)(V+1)*sizeof(int)
                     + (size_t)E*sizeof(int)
                     + (size_t)V*sizeof(int)
                     + (size_t)V*sizeof(int)
                     + (size_t)V*sizeof(int)
                     + sizeof(int);
    double gpu_mb = gpu_bytes / (1024.0 * 1024.0);

    size_t cpu_bytes = (size_t)(V+1)*sizeof(int)
                     + (size_t)E*sizeof(int)
                     + (size_t)V*sizeof(int);
    double cpu_mb = cpu_bytes / (1024.0 * 1024.0);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    cudaEventRecord(start);

    bfs_cuda(V, E, row_ptr, col_idx, 0, h_level);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    float ms = 0;
    cudaEventElapsedTime(&ms, start, stop);

    int visited = 0;
    for (int i = 0; i < V; i++) if (h_level[i] != -1) visited++;

    printf("=============================\n");
    printf("  Dataset     : %s\n",      label);
    printf("  Nodes       : %d\n",      V);
    printf("  Edges       : %d\n",      E);
    printf("  Visited     : %d\n",      visited);
    printf("  Time        : %.3f ms\n", ms);
    printf("  GPU Mem     : %.2f MB\n", gpu_mb);
    printf("  CPU Mem     : %.2f MB\n", cpu_mb);
    printf("  Total Mem   : %.2f MB\n", gpu_mb + cpu_mb);
    printf("  Throughput  : %.0f edges/ms\n", E / ms);
    printf("=============================\n\n");

    cudaEventDestroy(start); cudaEventDestroy(stop);
    free(row_ptr); free(col_idx); free(h_level);
}

int main(void) {
    run_test("Small",   1000,    8);
    run_test("Medium",  64000,   8);
    run_test("Large",   256000,  8);
    run_test("XLarge",  1000000, 8);
    return 0;
}

Writing bfs_cuda.cu


In [ ]:
!nvidia-smi

Sat Apr 18 21:41:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             43W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
!nvcc -O2 -arch=sm_80 bfs_cuda.cu -o bfs_cuda

In [ ]:
!./bfs_cuda

  Dataset     : Small
  Nodes       : 1000
  Edges       : 7992
  Visited     : 999
  Time        : 1.400 ms
  GPU Mem     : 0.05 MB
  CPU Mem     : 0.04 MB
  Total Mem   : 0.08 MB
  Throughput  : 5708 edges/ms

  Dataset     : Medium
  Nodes       : 64000
  Edges       : 511993
  Visited     : 63985
  Time        : 1.462 ms
  GPU Mem     : 2.93 MB
  CPU Mem     : 2.44 MB
  Total Mem   : 5.37 MB
  Throughput  : 350128 edges/ms

  Dataset     : Large
  Nodes       : 256000
  Edges       : 2047993
  Visited     : 255910
  Time        : 3.699 ms
  GPU Mem     : 11.72 MB
  CPU Mem     : 9.77 MB
  Total Mem   : 21.48 MB
  Throughput  : 553612 edges/ms

  Dataset     : XLarge
  Nodes       : 1000000
  Edges       : 7999994
  Visited     : 999652
  Time        : 11.760 ms
  GPU Mem     : 45.78 MB
  CPU Mem     : 38.15 MB
  Total Mem   : 83.92 MB
  Throughput  : 680292 edges/ms



In [ ]:
#bfs_open_cl

In [ ]:
%%writefile bfs_opencl.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>
#define CL_TARGET_OPENCL_VERSION 200
#define CL_USE_DEPRECATED_OPENCL_1_2_APIS
#include <CL/cl.h>

#define INF -1

static double now_ms(void) {
    struct timespec ts;
    clock_gettime(CLOCK_MONOTONIC, &ts);
    return ts.tv_sec * 1e3 + ts.tv_nsec * 1e-6;
}

const char* BFS_KERNEL_SRC =
"__kernel void bfs_kernel(                                          \n"
"    __global const int* row_ptr,                                   \n"
"    __global const int* col_idx,                                   \n"
"    __global int*       level,                                     \n"
"    __global const int* frontier,                                  \n"
"    __global int*       next_frontier,                             \n"
"    __global int*       next_size,                                 \n"
"    int                 frontier_size,                             \n"
"    int                 current_level)                             \n"
"{                                                                  \n"
"    int tid = get_global_id(0);                                    \n"
"    if (tid >= frontier_size) return;                              \n"
"    int u = frontier[tid];                                         \n"
"    for (int e = row_ptr[u]; e < row_ptr[u + 1]; e++) {           \n"
"        int v = col_idx[e];                                        \n"
"        int old = atomic_cmpxchg(&level[v], -1, current_level+1); \n"
"        if (old == -1) {                                           \n"
"            int pos = atomic_inc(next_size);                       \n"
"            next_frontier[pos] = v;                                \n"
"        }                                                          \n"
"    }                                                              \n"
"}                                                                  \n";

void bfs_opencl(int V, int E, const int* h_row_ptr, const int* h_col_idx,
                int source, int* h_level, double* out_ms)
{
    cl_int err;
    cl_platform_id platform;
    cl_device_id   device;
    clGetPlatformIDs(1, &platform, NULL);

    err = clGetDeviceIDs(platform, CL_DEVICE_TYPE_GPU, 1, &device, NULL);
    if (err != CL_SUCCESS) {
        printf("No GPU found, using CPU...\n");
        err = clGetDeviceIDs(platform, CL_DEVICE_TYPE_CPU, 1, &device, NULL);
    }

    char devname[128];
    clGetDeviceInfo(device, CL_DEVICE_NAME, 128, devname, NULL);
    printf("Device: %s\n", devname);

    cl_context ctx = clCreateContext(NULL, 1, &device, NULL, NULL, &err);

    #pragma GCC diagnostic push
    #pragma GCC diagnostic ignored "-Wdeprecated-declarations"

    cl_command_queue queue = clCreateCommandQueue(ctx, device, CL_QUEUE_PROFILING_ENABLE, &err);
    #pragma GCC diagnostic pop

    cl_program prog = clCreateProgramWithSource(ctx, 1, &BFS_KERNEL_SRC, NULL, &err);
    err = clBuildProgram(prog, 1, &device, NULL, NULL, NULL);
    if (err != CL_SUCCESS) {
        size_t log_size;
        clGetProgramBuildInfo(prog, device, CL_PROGRAM_BUILD_LOG, 0, NULL, &log_size);
        char* log = (char*)malloc(log_size);
        clGetProgramBuildInfo(prog, device, CL_PROGRAM_BUILD_LOG, log_size, log, NULL);
        fprintf(stderr, "Build error:\n%s\n", log);
        free(log); exit(1);
    }
    cl_kernel kernel = clCreateKernel(prog, "bfs_kernel", &err);

    for (int i = 0; i < V; i++) h_level[i] = INF;
    h_level[source] = 0;

    double t_start = now_ms();

    cl_mem d_row_ptr       = clCreateBuffer(ctx, CL_MEM_READ_ONLY|CL_MEM_COPY_HOST_PTR,  (V+1)*sizeof(int), (void*)h_row_ptr, &err);
    cl_mem d_col_idx       = clCreateBuffer(ctx, CL_MEM_READ_ONLY|CL_MEM_COPY_HOST_PTR,  E*sizeof(int),     (void*)h_col_idx, &err);
    cl_mem d_level         = clCreateBuffer(ctx, CL_MEM_READ_WRITE|CL_MEM_COPY_HOST_PTR, V*sizeof(int),     h_level, &err);
    cl_mem d_frontier      = clCreateBuffer(ctx, CL_MEM_READ_WRITE, V*sizeof(int), NULL, &err);
    cl_mem d_next_frontier = clCreateBuffer(ctx, CL_MEM_READ_WRITE, V*sizeof(int), NULL, &err);
    cl_mem d_next_size     = clCreateBuffer(ctx, CL_MEM_READ_WRITE, sizeof(int),   NULL, &err);

    clEnqueueWriteBuffer(queue, d_frontier, CL_TRUE, 0, sizeof(int), &source, 0, NULL, NULL);

    int h_frontier_size = 1;

    for (int lvl = 0; h_frontier_size > 0; lvl++) {
        int zero = 0;
        clEnqueueWriteBuffer(queue, d_next_size, CL_TRUE, 0, sizeof(int), &zero, 0, NULL, NULL);

        clSetKernelArg(kernel, 0, sizeof(cl_mem), &d_row_ptr);
        clSetKernelArg(kernel, 1, sizeof(cl_mem), &d_col_idx);
        clSetKernelArg(kernel, 2, sizeof(cl_mem), &d_level);
        clSetKernelArg(kernel, 3, sizeof(cl_mem), &d_frontier);
        clSetKernelArg(kernel, 4, sizeof(cl_mem), &d_next_frontier);
        clSetKernelArg(kernel, 5, sizeof(cl_mem), &d_next_size);
        clSetKernelArg(kernel, 6, sizeof(int),    &h_frontier_size);
        clSetKernelArg(kernel, 7, sizeof(int),    &lvl);

        size_t gws = ((h_frontier_size + 255) / 256) * 256;
        size_t lws = 256;

        clEnqueueNDRangeKernel(queue, kernel, 1, NULL, &gws, &lws, 0, NULL, NULL);
        clFinish(queue);

        clEnqueueReadBuffer(queue, d_next_size, CL_TRUE, 0, sizeof(int), &h_frontier_size, 0, NULL, NULL);

        cl_mem tmp      = d_frontier;
        d_frontier      = d_next_frontier;
        d_next_frontier = tmp;
    }

      clEnqueueReadBuffer(queue, d_level, CL_TRUE, 0, V*sizeof(int), h_level, 0, NULL, NULL);

    *out_ms = now_ms() - t_start;

    clReleaseMemObject(d_row_ptr); clReleaseMemObject(d_col_idx);
    clReleaseMemObject(d_level);   clReleaseMemObject(d_frontier);
    clReleaseMemObject(d_next_frontier); clReleaseMemObject(d_next_size);
    clReleaseKernel(kernel); clReleaseProgram(prog);
    clReleaseCommandQueue(queue); clReleaseContext(ctx);
}

void generate_graph(int V, int avg_degree, int** out_row_ptr, int** out_col_idx, int* out_E) {
    srand(42);
    int** adj    = (int**)calloc(V, sizeof(int*));
    int*  adj_sz = (int*)calloc(V, sizeof(int));
    for (int u = 0; u < V; u++) {
        adj[u] = (int*)malloc(avg_degree * 2 * sizeof(int));
        for (int i = 0; i < avg_degree; i++) {
            int v = rand() % V;
            if (v != u) adj[u][adj_sz[u]++] = v;
        }
    }
    int* row_ptr = (int*)calloc(V + 1, sizeof(int));
    for (int u = 0; u < V; u++) row_ptr[u+1] = row_ptr[u] + adj_sz[u];
    int E = row_ptr[V];
    int* col_idx = (int*)malloc(E * sizeof(int));
    for (int u = 0; u < V; u++)
        for (int i = 0; i < adj_sz[u]; i++)
            col_idx[row_ptr[u] + i] = adj[u][i];
    for (int u = 0; u < V; u++) free(adj[u]);
    free(adj); free(adj_sz);
    *out_row_ptr = row_ptr; *out_col_idx = col_idx; *out_E = E;
}

void run_test(const char* label, int V, int avg_degree) {
    int *row_ptr, *col_idx, E;
    generate_graph(V, avg_degree, &row_ptr, &col_idx, &E);
    int* h_level = (int*)malloc(V * sizeof(int));
    double ms = 0.0;

    bfs_opencl(V, E, row_ptr, col_idx, 0, h_level, &ms);

    size_t gpu_bytes = (size_t)(V+1)*sizeof(int) + (size_t)E*sizeof(int)
                     + (size_t)V*sizeof(int)*3 + sizeof(int);
    double gpu_mb = gpu_bytes / (1024.0 * 1024.0);

    size_t cpu_bytes = (size_t)(V+1)*sizeof(int) + (size_t)E*sizeof(int)
                     + (size_t)V*sizeof(int);
    double cpu_mb = cpu_bytes / (1024.0 * 1024.0);

    int visited = 0;
    for (int i = 0; i < V; i++) if (h_level[i] != -1) visited++;

    printf("=============================\n");
    printf("  Dataset     : %s\n",      label);
    printf("  Nodes       : %d\n",      V);
    printf("  Edges       : %d\n",      E);
    printf("  Visited     : %d\n",      visited);
    printf("  Time        : %.3f ms\n", ms);
    printf("  GPU Mem     : %.2f MB\n", gpu_mb);   /* matches CUDA label */
    printf("  CPU Mem     : %.2f MB\n", cpu_mb);   /* matches CUDA label */
    printf("  Total Mem   : %.2f MB\n", gpu_mb + cpu_mb);
    printf("  Throughput  : %.0f edges/ms\n", (double)E / ms);
    printf("=============================\n\n");

    free(row_ptr); free(col_idx); free(h_level);
}

int main(void) {
    run_test("Small",   1000,    8);
    run_test("Medium",  64000,   8);
    run_test("Large",   256000,  8);
    run_test("XLarge",  1000000, 8);
    return 0;
}

Overwriting bfs_opencl.c


In [ ]:
!apt-get install -y pocl-opencl-icd ocl-icd-opencl-dev opencl-headers -q

Reading package lists...
Building dependency tree...
Reading state information...
ocl-icd-opencl-dev is already the newest version (2.2.14-3).
opencl-headers is already the newest version (3.0~2022.01.04-1).
pocl-opencl-icd is already the newest version (1.8-3).
0 upgraded, 0 newly installed, 0 to remove and 39 not upgraded.


In [ ]:
!gcc -O2 -o bfs_opencl bfs_opencl.c -lOpenCL && ./bfs_opencl

Device: NVIDIA A100-SXM4-40GB
  Dataset     : Small
  Nodes       : 1000
  Edges       : 7992
  Visited     : 999
  Time        : 0.871 ms
  GPU Mem     : 0.05 MB
  CPU Mem     : 0.04 MB
  Total Mem   : 0.08 MB
  Throughput  : 9171 edges/ms

Device: NVIDIA A100-SXM4-40GB
  Dataset     : Medium
  Nodes       : 64000
  Edges       : 511993
  Visited     : 63985
  Time        : 2.292 ms
  GPU Mem     : 2.93 MB
  CPU Mem     : 2.44 MB
  Total Mem   : 5.37 MB
  Throughput  : 223425 edges/ms

Device: NVIDIA A100-SXM4-40GB
  Dataset     : Large
  Nodes       : 256000
  Edges       : 2047993
  Visited     : 255910
  Time        : 9.517 ms
  GPU Mem     : 11.72 MB
  CPU Mem     : 9.77 MB
  Total Mem   : 21.48 MB
  Throughput  : 215201 edges/ms

Device: NVIDIA A100-SXM4-40GB
  Dataset     : XLarge
  Nodes       : 1000000
  Edges       : 7999994
  Visited     : 999652
  Time        : 19.396 ms
  GPU Mem     : 45.78 MB
  CPU Mem     : 38.15 MB
  Total Mem   : 83.92 MB
  Throughput  : 412459 edges/

In [ ]:
#PAGERANK_CPU

In [ ]:
%%writefile page_rank.c

#include <iostream>
#include <fstream>
#include <sstream>
#include <vector>
#include <cmath>
#include <chrono>
#include <random>
#include <numeric>
#include <algorithm>
#include <iomanip>
#include <string>


static const double DAMPING  = 0.85;
static const double EPSILON  = 1e-6;
static const int    MAX_ITER = 200;

//CSR Representation
struct Graph {
    int              n;           // number of nodes
    long             m;           // number of edges
    std::vector<int> out_deg;     // out_deg[u]  = # edges leaving node u
    std::vector<int> row_ptr;     // out-edge CSR: node u's edges are in dst[row_ptr[u]..row_ptr[u+1])
    std::vector<int> dst;         // out-edge CSR: destination node IDs
    std::vector<int> in_row_ptr;  // in-edge  CSR: node v's in-edges are in in_dst[in_row_ptr[v]..in_row_ptr[v+1])
    std::vector<int> in_dst;      // in-edge  CSR: source node IDs
};

void build_csr(Graph& g, std::vector<std::pair<int,int>>& edges) {
    g.m = (long)edges.size();
    g.out_deg.assign(g.n, 0);


    for (auto& [u, v] : edges) g.out_deg[u]++;

    g.row_ptr.resize(g.n + 1, 0);
    for (int i = 0; i < g.n; i++)
        g.row_ptr[i + 1] = g.row_ptr[i] + g.out_deg[i];

    g.dst.resize(g.m);
    std::vector<int> pos(g.row_ptr.begin(), g.row_ptr.end());
    for (auto& [u, v] : edges) g.dst[pos[u]++] = v;

    // ── In-edge CSR (transpose graph) ────────────────────────
    std::vector<int> in_deg(g.n, 0);
    for (auto& [u, v] : edges) in_deg[v]++;

    g.in_row_ptr.resize(g.n + 1, 0);
    for (int i = 0; i < g.n; i++)
        g.in_row_ptr[i + 1] = g.in_row_ptr[i] + in_deg[i];

    g.in_dst.resize(g.m);
    std::vector<int> ipos(g.in_row_ptr.begin(), g.in_row_ptr.end());
    for (auto& [u, v] : edges) g.in_dst[ipos[v]++] = u;
}

Graph build_random_graph(int n, long target_edges, unsigned seed = 42) {
    Graph g;
    g.n = n;

    std::mt19937 rng(seed);
    std::uniform_int_distribution<int> dist(0, n - 1);

    std::vector<std::pair<int,int>> edges;
    edges.reserve(target_edges);

    long attempts = 0;
    while ((long)edges.size() < target_edges && attempts < target_edges * 4) {
        int u = dist(rng), v = dist(rng);
        if (u != v) edges.push_back({u, v});
        attempts++;
    }

    build_csr(g, edges);
    return g;
}

Graph load_snap_graph(const std::string& filename) {
    std::ifstream file(filename);
    if (!file.is_open()) {
        std::cerr << "\nERROR: Cannot open file: " << filename << "\n";
        std::cerr << "Steps to fix:\n";
        std::cerr << "  1. Download: wget https://snap.stanford.edu/data/web-Stanford.txt.gz\n";
        std::cerr << "  2. Unzip:    gunzip web-Stanford.txt.gz\n";
        std::cerr << "  3. Run:      ./pagerank_cpu web-Stanford.txt\n\n";
        std::exit(1);
    }

    std::cout << "Reading: " << filename << " ...\n";

    std::vector<std::pair<int,int>> edges;
    std::string line;
    int  max_node = 0;
    long self_loops = 0;
    long bad_lines  = 0;

    while (std::getline(file, line)) {
        if (line.empty() || line[0] == '#') continue;   // skip headers/comments

        std::istringstream ss(line);
        int u, v;
        if (!(ss >> u >> v)) { bad_lines++; continue; } // skip malformed lines

        if (u < 0 || v < 0)  { bad_lines++; continue; } // skip invalid IDs
        if (u == v) { self_loops++; continue; }          // skip self-loops

        edges.push_back({u, v});
        max_node = std::max(max_node, std::max(u, v));
    }

    std::cout << "  Edges loaded      : " << edges.size()  << "\n";
    std::cout << "  Self-loops skipped: " << self_loops    << "\n";
    if (bad_lines > 0)
        std::cout << "  Malformed lines   : " << bad_lines << "\n";
    std::cout << "  Max node ID       : " << max_node      << "\n";
    std::cout << "  Total nodes       : " << max_node + 1  << "\n\n";

    Graph g;
    g.n = max_node + 1;   // nodes are 0-indexed
    build_csr(g, edges);
    return g;
}


struct PRResult {
    std::vector<double> rank;   // final score per node
    int    iterations;          // rounds until convergence
    double max_diff;            // final max rank change
    double time_ms;             // wall-clock time in ms
    double mem_mb;              // estimated memory in MB
};

PRResult pagerank_cpu(const Graph& g) {
    int n = g.n;

    // Estimate memory footprint
    double mem_mb = (double)(
        n * 2 * sizeof(double) +      // rank[] + new_rank[]
        g.m * sizeof(int) * 2 +       // dst[]  + in_dst[]
        (n + 1) * sizeof(int) * 2 +   // row_ptr[] + in_row_ptr[]
        n * sizeof(int)               // out_deg[]
    ) / (1024.0 * 1024.0);

    // Initialise: every node gets equal rank (sum = 1.0)
    std::vector<double> rank(n, 1.0 / n);
    std::vector<double> new_rank(n, 0.0);

    const double teleport = (1.0 - DAMPING) / n;

    auto t0 = std::chrono::high_resolution_clock::now();

    int    iter = 0;
    double diff = 1.0;   // set > EPSILON so loop enters

    while (diff > EPSILON && iter < MAX_ITER) {


        double dangling_sum = 0.0;
        for (int i = 0; i < n; i++)
            if (g.out_deg[i] == 0)
                dangling_sum += rank[i];
        double dangling_contrib = DAMPING * dangling_sum / n;


        for (int i = 0; i < n; i++)
            new_rank[i] = teleport + dangling_contrib;

        // (u splits its rank equally among all out-links)
        for (int v = 0; v < n; v++) {
            for (int k = g.in_row_ptr[v]; k < g.in_row_ptr[v + 1]; k++) {
                int u = g.in_dst[k];
                new_rank[v] += DAMPING * rank[u] / g.out_deg[u];
            }
        }

        // ── Step 4: Check convergence ────────────────────────
        // If no node's rank changed by more than EPSILON, we're done
        diff = 0.0;
        for (int i = 0; i < n; i++)
            diff = std::max(diff, std::abs(new_rank[i] - rank[i]));

        std::swap(rank, new_rank);   // O(1) pointer flip — no data copy
        iter++;
    }

    auto t1 = std::chrono::high_resolution_clock::now();
    double ms = std::chrono::duration<double, std::milli>(t1 - t0).count();

    return { rank, iter, diff, ms, mem_mb };
}


void print_results(const std::string& label, const Graph& g, const PRResult& r) {
    double throughput = (double)g.m * r.iterations / r.time_ms;

    // Find top-5 nodes by PageRank
    int top_k = std::min(5, g.n);
    std::vector<int> idx(g.n);
    std::iota(idx.begin(), idx.end(), 0);
    std::partial_sort(idx.begin(), idx.begin() + top_k, idx.end(),
        [&](int a, int b){ return r.rank[a] > r.rank[b]; });

    std::cout << "=============================\n";
    std::cout << "  Dataset     : " << label    << "\n";
    std::cout << "  Nodes       : " << g.n      << "\n";
    std::cout << "  Edges       : " << g.m      << "\n";
    std::cout << "  Iterations  : " << r.iterations << "\n";
    std::cout << "  Converged   : "
              << (r.max_diff < EPSILON ? "Yes" : "No — hit MAX_ITER cap")
              << "  (max|diff| = "
              << std::scientific << std::setprecision(2) << r.max_diff << ")\n";
    std::cout << std::fixed << std::setprecision(6);
    std::cout << "  Time        : " << r.time_ms << " ms\n";
    std::cout << "  CPU Mem     : " << r.mem_mb  << " MB\n";
    std::cout << "  GPU Mem     : 0.00 MB (CPU only)\n";
    std::cout << "  Total Mem   : " << r.mem_mb  << " MB\n";
    std::cout << std::fixed << std::setprecision(0);
    std::cout << "  Throughput  : " << throughput << " edge-iters/ms\n";
    std::cout << "  Top-5 Nodes :\n";
    for (int i = 0; i < top_k; i++) {
        std::cout << "    Rank " << (i + 1) << " → Node #"
                  << std::setw(8) << idx[i]
                  << "   PR = " << std::scientific << std::setprecision(6)
                  << r.rank[idx[i]] << "\n";
    }
    std::cout << "=============================\n\n";
}


int main() {

    std::cout << "==============================================\n";
    std::cout << "  CPU PageRank Benchmark\n";
    std::cout << "  d=" << DAMPING
              << "  eps=" << EPSILON
              << "  max_iter=" << MAX_ITER << "\n";
    std::cout << "==============================================\n\n";

    // ── PART 1: Synthetic random graphs ──────────────────────
    std::cout << "--------------------------------------------\n";
    std::cout << "  PART 1 — Synthetic Random Graphs\n";
    std::cout << "--------------------------------------------\n\n";

    struct Dataset { std::string name; int n; long m; };
    std::vector<Dataset> datasets = {
        { "Small",   1000,     7992    },
        { "Medium",  64000,    511993  },
        { "Large",   256000,   2047993 },
        { "XLarge",  1000000,  7999994 },
    };

    for (auto& d : datasets) {
        std::cout << "Building " << d.name << " graph ("
                  << d.n << " nodes, " << d.m << " edges)...\n";
        Graph    g = build_random_graph(d.n, d.m);
        PRResult r = pagerank_cpu(g);
        print_results(d.name, g, r);
    }

    // ── PART 2: Real-world SNAP graph ────────────────────────
    std::cout << "--------------------------------------------\n";
    std::cout << "  PART 2 — Real-World SNAP Dataset\n";
    std::cout << "--------------------------------------------\n\n";

    Graph    g = load_snap_graph("web-Stanford.txt");
    PRResult r = pagerank_cpu(g);
    print_results("web-Stanford", g, r);

    return 0;
}

Overwriting page_rank.c


In [ ]:
!g++ -O2 -std=c++17 -o pagerank_cpu page_rank.c && echo "Compiled successfully!"

Compiled successfully!


In [ ]:
!./pagerank_cpu

  CPU PageRank Benchmark
  d=0.85  eps=1e-06  max_iter=200

--------------------------------------------
  PART 1 — Synthetic Random Graphs
--------------------------------------------

Building Small graph (1000 nodes, 7992 edges)...
  Dataset     : Small
  Nodes       : 1000
  Edges       : 7992
  Iterations  : 8
  Converged   : Yes  (max|diff| = 9.83e-07)
  Time        : 0.246115 ms
  CPU Mem     : 0.087685 MB
  GPU Mem     : 0.00 MB (CPU only)
  Total Mem   : 0.087685 MB
  Throughput  : 259781 edge-iters/ms
  Top-5 Nodes :
    Rank 1 → Node #     609   PR = 2.650276e-03
    Rank 2 → Node #     994   PR = 2.552684e-03
    Rank 3 → Node #      46   PR = 2.408849e-03
    Rank 4 → Node #     438   PR = 2.265941e-03
    Rank 5 → Node #       2   PR = 2.238152e-03

Building Medium graph (64000 nodes, 511993 edges)...
  Dataset     : Medium
  Nodes       : 64000
  Edges       : 511993
  Iterations  : 5
  Converged   : Yes  (max|diff| = 4.78e-07)
  Time        : 9.047733 ms
  CPU Mem     :

In [ ]:
%%writefile pagerank_cuda.cu

#include <iostream>
#include <fstream>
#include <sstream>
#include <vector>
#include <cmath>
#include <chrono>
#include <random>
#include <numeric>
#include <algorithm>
#include <iomanip>
#include <string>
#include <cuda_runtime.h>

// ── Parameters ──────────────────────────────────────────────
static const double DAMPING  = 0.85;
static const double EPSILON  = 1e-6;
static const int    MAX_ITER = 200;

// ── CUDA error check macro ───────────────────────────────────
#define CUDA_CHECK(call)                                                \
    do {                                                                \
        cudaError_t err = call;                                         \
        if (err != cudaSuccess) {                                       \
            std::cerr << "CUDA error at " << __FILE__ << ":" << __LINE__\
                      << " — " << cudaGetErrorString(err) << "\n";     \
            std::exit(1);                                               \
        }                                                               \
    } while (0)

// ════════════════════════════════════════════════════════════
//  Graph — CSR (host side)
// ════════════════════════════════════════════════════════════
struct Graph {
    int              n;
    long             m;
    std::vector<int> out_deg;
    std::vector<int> row_ptr;     // out-edge CSR
    std::vector<int> dst;
    std::vector<int> in_row_ptr;  // in-edge CSR
    std::vector<int> in_dst;
};

void build_csr(Graph& g, std::vector<std::pair<int,int>>& edges) {
    g.m = (long)edges.size();
    g.out_deg.assign(g.n, 0);

    for (auto& [u, v] : edges) g.out_deg[u]++;

    g.row_ptr.resize(g.n + 1, 0);
    for (int i = 0; i < g.n; i++)
        g.row_ptr[i + 1] = g.row_ptr[i] + g.out_deg[i];
    g.dst.resize(g.m);
    std::vector<int> pos(g.row_ptr.begin(), g.row_ptr.end());
    for (auto& [u, v] : edges) g.dst[pos[u]++] = v;

    std::vector<int> in_deg(g.n, 0);
    for (auto& [u, v] : edges) in_deg[v]++;
    g.in_row_ptr.resize(g.n + 1, 0);
    for (int i = 0; i < g.n; i++)
        g.in_row_ptr[i + 1] = g.in_row_ptr[i] + in_deg[i];
    g.in_dst.resize(g.m);
    std::vector<int> ipos(g.in_row_ptr.begin(), g.in_row_ptr.end());
    for (auto& [u, v] : edges) g.in_dst[ipos[v]++] = u;
}

Graph build_random_graph(int n, long target_edges, unsigned seed = 42) {
    Graph g; g.n = n;
    std::mt19937 rng(seed);
    std::uniform_int_distribution<int> dist(0, n - 1);
    std::vector<std::pair<int,int>> edges;
    edges.reserve(target_edges);
    long attempts = 0;
    while ((long)edges.size() < target_edges && attempts < target_edges * 4) {
        int u = dist(rng), v = dist(rng);
        if (u != v) edges.push_back({u, v});
        attempts++;
    }
    build_csr(g, edges);
    return g;
}

Graph load_snap_graph(const std::string& filename) {
    std::ifstream file(filename);
    if (!file.is_open()) {
        std::cerr << "ERROR: Cannot open file: " << filename << "\n";
        std::exit(1);
    }
    std::cout << "Reading: " << filename << " ...\n";
    std::vector<std::pair<int,int>> edges;
    std::string line;
    int max_node = 0;
    long self_loops = 0;
    while (std::getline(file, line)) {
        if (line.empty() || line[0] == '#') continue;
        std::istringstream ss(line);
        int u, v;
        if (!(ss >> u >> v)) continue;
        if (u == v) { self_loops++; continue; }
        edges.push_back({u, v});
        max_node = std::max(max_node, std::max(u, v));
    }
    std::cout << "  Edges loaded      : " << edges.size() << "\n";
    std::cout << "  Self-loops skipped: " << self_loops   << "\n";
    std::cout << "  Total nodes       : " << max_node + 1 << "\n\n";
    Graph g; g.n = max_node + 1;
    build_csr(g, edges);
    return g;
}


// Kernel 1: compute dangling sum contribution using atomicAdd
__global__ void kernel_dangling_sum(
    const double* __restrict__ rank,
    const int*    __restrict__ out_deg,
    double* dangling_sum,
    int n)
{
    int v = blockIdx.x * blockDim.x + threadIdx.x;
    if (v >= n) return;
    if (out_deg[v] == 0)
        atomicAdd(dangling_sum, rank[v]);
}

// Kernel 2: initialise new_rank with teleport + dangling base
__global__ void kernel_init_rank(
    double* new_rank,
    double  teleport,
    double  dangling_contrib,
    int     n)
{
    int v = blockIdx.x * blockDim.x + threadIdx.x;
    if (v >= n) return;
    new_rank[v] = teleport + dangling_contrib;
}

// Kernel 3: scatter rank through in-edges (one thread per node)
__global__ void kernel_scatter(
    const double* __restrict__ rank,
    const int*    __restrict__ out_deg,
    const int*    __restrict__ in_row_ptr,
    const int*    __restrict__ in_dst,
    double*       new_rank,
    double        damping,
    int           n)
{
    int v = blockIdx.x * blockDim.x + threadIdx.x;
    if (v >= n) return;

    double contrib = 0.0;
    int start = in_row_ptr[v];
    int end   = in_row_ptr[v + 1];
    for (int k = start; k < end; k++) {
        int u = in_dst[k];
        contrib += rank[u] / (double)out_deg[u];
    }
    new_rank[v] += damping * contrib;
}

// Kernel 4: compute max|diff| using atomicMax (via integer trick)
__global__ void kernel_diff(
    const double* __restrict__ rank,
    const double* __restrict__ new_rank,
    unsigned long long* max_diff_bits,
    int n)
{
    int v = blockIdx.x * blockDim.x + threadIdx.x;
    if (v >= n) return;

    double diff = fabs(new_rank[v] - rank[v]);
    // Store double as ULL for atomicMax (works because IEEE 754 doubles
    // are monotone in their bit representation for positive values)
    unsigned long long bits;
    memcpy(&bits, &diff, sizeof(double));
    atomicMax(max_diff_bits, bits);
}

// ════════════════════════════════════════════════════════════
//  Device memory helper
// ════════════════════════════════════════════════════════════
struct DeviceGraph {
    int*    out_deg    = nullptr;
    int*    in_row_ptr = nullptr;
    int*    in_dst     = nullptr;
    int n; long m;

    void alloc_and_copy(const Graph& g) {
        n = g.n; m = g.m;
        CUDA_CHECK(cudaMalloc(&out_deg,    n * sizeof(int)));
        CUDA_CHECK(cudaMalloc(&in_row_ptr, (n + 1) * sizeof(int)));
        CUDA_CHECK(cudaMalloc(&in_dst,     m * sizeof(int)));
        CUDA_CHECK(cudaMemcpy(out_deg,    g.out_deg.data(),    n * sizeof(int),     cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(in_row_ptr, g.in_row_ptr.data(), (n+1) * sizeof(int), cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(in_dst,     g.in_dst.data(),     m * sizeof(int),     cudaMemcpyHostToDevice));
    }

    void free_all() {
        cudaFree(out_deg); cudaFree(in_row_ptr); cudaFree(in_dst);
    }
};


struct PRResult {
    std::vector<double> rank;
    int    iterations;
    double max_diff;
    double time_ms;
    double cpu_mem_mb;
    double gpu_mem_mb;
};

PRResult pagerank_cuda(const Graph& g) {
    int  n = g.n;
    long m = g.m;

    double cpu_mem_mb = (double)(
        n * sizeof(double) +        // host rank copy
        n * sizeof(int) * 3 +       // out_deg, in_row_ptr, in_dst (host)
        m * sizeof(int)
    ) / (1024.0 * 1024.0);

    double gpu_mem_mb = (double)(
        n * 2 * sizeof(double) +    // rank[] + new_rank[]
        n * sizeof(int) +           // out_deg
        (n + 1) * sizeof(int) +     // in_row_ptr
        m * sizeof(int)             // in_dst
    ) / (1024.0 * 1024.0);

    // ── Upload graph to GPU ──────────────────────────────────
    DeviceGraph dg;
    dg.alloc_and_copy(g);

    // ── Allocate rank arrays on GPU ──────────────────────────
    double *d_rank, *d_new_rank;
    CUDA_CHECK(cudaMalloc(&d_rank,     n * sizeof(double)));
    CUDA_CHECK(cudaMalloc(&d_new_rank, n * sizeof(double)));

    // ── Initialise rank to 1/n on GPU ───────────────────────
    std::vector<double> h_rank(n, 1.0 / n);
    CUDA_CHECK(cudaMemcpy(d_rank, h_rank.data(), n * sizeof(double), cudaMemcpyHostToDevice));

    // ── Dangling sum + diff accumulators ─────────────────────
    double*            d_dangling_sum;
    unsigned long long* d_max_diff_bits;
    CUDA_CHECK(cudaMalloc(&d_dangling_sum,  sizeof(double)));
    CUDA_CHECK(cudaMalloc(&d_max_diff_bits, sizeof(unsigned long long)));

    const double teleport = (1.0 - DAMPING) / n;
    const int    threads  = 256;
    const int    blocks   = (n + threads - 1) / threads;

    // ── Timing ──────────────────────────────────────────────
    cudaEvent_t t_start, t_stop;
    CUDA_CHECK(cudaEventCreate(&t_start));
    CUDA_CHECK(cudaEventCreate(&t_stop));
    CUDA_CHECK(cudaEventRecord(t_start));

    int    iter = 0;
    double diff = 1.0;

    while (diff > EPSILON && iter < MAX_ITER) {

        // Step 1: dangling sum
        double zero = 0.0;
        CUDA_CHECK(cudaMemcpy(d_dangling_sum, &zero, sizeof(double), cudaMemcpyHostToDevice));
        kernel_dangling_sum<<<blocks, threads>>>(d_rank, dg.out_deg, d_dangling_sum, n);

        double h_dangling_sum;
        CUDA_CHECK(cudaMemcpy(&h_dangling_sum, d_dangling_sum, sizeof(double), cudaMemcpyDeviceToHost));
        double dangling_contrib = DAMPING * h_dangling_sum / n;

        // Step 2: initialise new_rank
        kernel_init_rank<<<blocks, threads>>>(d_new_rank, teleport, dangling_contrib, n);

        // Step 3: scatter rank through in-edges
        kernel_scatter<<<blocks, threads>>>(
            d_rank, dg.out_deg, dg.in_row_ptr, dg.in_dst,
            d_new_rank, DAMPING, n);

        // Step 4: compute convergence
        unsigned long long zero_ull = 0ULL;
        CUDA_CHECK(cudaMemcpy(d_max_diff_bits, &zero_ull, sizeof(unsigned long long), cudaMemcpyHostToDevice));
        kernel_diff<<<blocks, threads>>>(d_rank, d_new_rank, d_max_diff_bits, n);

        unsigned long long h_bits;
        CUDA_CHECK(cudaMemcpy(&h_bits, d_max_diff_bits, sizeof(unsigned long long), cudaMemcpyDeviceToHost));
        memcpy(&diff, &h_bits, sizeof(double));

        // Step 5: swap pointers
        std::swap(d_rank, d_new_rank);
        iter++;
    }

    CUDA_CHECK(cudaEventRecord(t_stop));
    CUDA_CHECK(cudaEventSynchronize(t_stop));
    float ms;
    CUDA_CHECK(cudaEventElapsedTime(&ms, t_start, t_stop));

    // ── Copy result back ─────────────────────────────────────
    CUDA_CHECK(cudaMemcpy(h_rank.data(), d_rank, n * sizeof(double), cudaMemcpyDeviceToHost));

    // ── Cleanup ──────────────────────────────────────────────
    dg.free_all();
    cudaFree(d_rank); cudaFree(d_new_rank);
    cudaFree(d_dangling_sum); cudaFree(d_max_diff_bits);
    cudaEventDestroy(t_start); cudaEventDestroy(t_stop);

    return { h_rank, iter, diff, (double)ms, cpu_mem_mb, gpu_mem_mb };
}

// ════════════════════════════════════════════════════════════
//  Print results
// ════════════════════════════════════════════════════════════
void print_results(const std::string& label, const Graph& g, const PRResult& r) {
    double throughput = (double)g.m * r.iterations / r.time_ms;
    int top_k = std::min(5, g.n);
    std::vector<int> idx(g.n);
    std::iota(idx.begin(), idx.end(), 0);
    std::partial_sort(idx.begin(), idx.begin() + top_k, idx.end(),
        [&](int a, int b){ return r.rank[a] > r.rank[b]; });

    std::cout << "=============================\n";
    std::cout << "  Dataset     : " << label          << "\n";
    std::cout << "  Nodes       : " << g.n            << "\n";
    std::cout << "  Edges       : " << g.m            << "\n";
    std::cout << "  Iterations  : " << r.iterations   << "\n";
    std::cout << "  Converged   : "
              << (r.max_diff < EPSILON ? "Yes" : "No (hit MAX_ITER)")
              << "  (max|diff| = "
              << std::scientific << std::setprecision(2) << r.max_diff << ")\n";
    std::cout << std::fixed << std::setprecision(6);
    std::cout << "  Time        : " << r.time_ms     << " ms\n";
    std::cout << "  GPU Mem     : " << r.gpu_mem_mb  << " MB\n";
    std::cout << "  CPU Mem     : " << r.cpu_mem_mb  << " MB\n";
    std::cout << "  Total Mem   : " << r.gpu_mem_mb + r.cpu_mem_mb << " MB\n";
    std::cout << std::fixed << std::setprecision(0);
    std::cout << "  Throughput  : " << throughput << " edge-iters/ms\n";
    std::cout << "  Top-5 Nodes :\n";
    for (int i = 0; i < top_k; i++) {
        std::cout << "    Rank " << (i+1) << " → Node #"
                  << std::setw(8) << idx[i]
                  << "   PR = " << std::scientific << std::setprecision(6)
                  << r.rank[idx[i]] << "\n";
    }
    std::cout << "=============================\n\n";
}

// ════════════════════════════════════════════════════════════
//  main
// ════════════════════════════════════════════════════════════
int main(int argc, char* argv[]) {

    // Print GPU info
    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
    std::cout << "==============================================\n";
    std::cout << "  CUDA PageRank Benchmark\n";
    std::cout << "  Device : " << prop.name << "\n";
    std::cout << "  d=" << DAMPING << "  eps=" << EPSILON
              << "  max_iter=" << MAX_ITER << "\n";
    std::cout << "==============================================\n\n";

    // ── SNAP file mode ───────────────────────────────────────
    if (argc == 2) {
        std::string filename = argv[1];
        Graph    g = load_snap_graph(filename);
        PRResult r = pagerank_cuda(g);
        std::string label = filename;
        size_t slash = label.find_last_of("/\\");
        if (slash != std::string::npos) label = label.substr(slash + 1);
        print_results(label, g, r);
        return 0;
    }

    // ── Synthetic benchmark mode ─────────────────────────────
    std::cout << "--------------------------------------------\n";
    std::cout << "  PART 1 — Synthetic Random Graphs\n";
    std::cout << "--------------------------------------------\n\n";

    struct Dataset { std::string name; int n; long m; };
    std::vector<Dataset> datasets = {
        { "Small",   1000,     7992    },
        { "Medium",  64000,    511993  },
        { "Large",   256000,   2047993 },
        { "XLarge",  1000000,  7999994 },
    };
    for (auto& d : datasets) {
        std::cout << "Building " << d.name << " graph...\n";
        Graph    g = build_random_graph(d.n, d.m);
        PRResult r = pagerank_cuda(g);
        print_results(d.name, g, r);
    }

    // ── SNAP web-Stanford ────────────────────────────────────
    std::cout << "--------------------------------------------\n";
    std::cout << "  PART 2 — Real-World SNAP Dataset\n";
    std::cout << "--------------------------------------------\n\n";
    Graph    g = load_snap_graph("web-Stanford.txt");
    PRResult r = pagerank_cuda(g);
    print_results("web-Stanford", g, r);

    return 0;
}

Overwriting pagerank_cuda.cu


In [ ]:
!nvcc -O2 -arch=sm_80 -o pagerank_cuda pagerank_cuda.cu


In [ ]:
!wget https://snap.stanford.edu/data/web-Stanford.txt.gz
!gunzip web-Stanford.txt.gz

--2026-04-21 21:26:57--  https://snap.stanford.edu/data/web-Stanford.txt.gz
Resolving snap.stanford.edu (snap.stanford.edu)... 171.64.75.80
Connecting to snap.stanford.edu (snap.stanford.edu)|171.64.75.80|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8908378 (8.5M) [application/x-gzip]
Saving to: ‘web-Stanford.txt.gz’

web-Stanford.txt.gz 100%[===================>]   8.50M  8.34MB/s    in 1.0s    

2026-04-21 21:26:58 (8.34 MB/s) - ‘web-Stanford.txt.gz’ saved [8908378/8908378]



In [ ]:
!./pagerank_cuda

  CUDA PageRank Benchmark
  Device : NVIDIA A100-SXM4-40GB
  d=0.85  eps=1e-06  max_iter=200

--------------------------------------------
  PART 1 — Synthetic Random Graphs
--------------------------------------------

Building Small graph...
  Dataset     : Small
  Nodes       : 1000
  Edges       : 7992
  Iterations  : 8
  Converged   : Yes  (max|diff| = 9.83e-07)
  Time        : 0.763648 ms
  GPU Mem     : 0.053379 MB
  CPU Mem     : 0.049561 MB
  Total Mem   : 0.102940 MB
  Throughput  : 83724 edge-iters/ms
  Top-5 Nodes :
    Rank 1 → Node #     609   PR = 2.650276e-03
    Rank 2 → Node #     994   PR = 2.552684e-03
    Rank 3 → Node #      46   PR = 2.408849e-03
    Rank 4 → Node #     438   PR = 2.265941e-03
    Rank 5 → Node #       2   PR = 2.238152e-03

Building Medium graph...
  Dataset     : Medium
  Nodes       : 64000
  Edges       : 511993
  Iterations  : 5
  Converged   : Yes  (max|diff| = 4.78e-07)
  Time        : 0.731552 ms
  GPU Mem     : 3.417946 MB
  CPU Mem     

In [ ]:
%%writefile pagerank_opencl.cpp
#ifdef __APPLE__
  #include <OpenCL/opencl.h>
#else
  #include <CL/cl.h>
#endif

#include <iostream>
#include <fstream>
#include <sstream>
#include <vector>
#include <cmath>
#include <chrono>
#include <random>
#include <numeric>
#include <algorithm>
#include <iomanip>
#include <string>
#include <cstring>

// ── Parameters ──────────────────────────────────────────────
static const double DAMPING  = 0.85;
static const double EPSILON  = 1e-6;
static const int    MAX_ITER = 200;

// ── OpenCL error check ───────────────────────────────────────
#define CL_CHECK(call)                                                  \
    do {                                                                \
        cl_int _err = (call);                                           \
        if (_err != CL_SUCCESS) {                                       \
            std::cerr << "OpenCL error " << _err                        \
                      << " at " << __FILE__ << ":" << __LINE__ << "\n"; \
            std::exit(1);                                               \
        }                                                               \
    } while (0)

//opencl kernel core
static const char* KERNEL_SRC = R"CL(
#pragma OPENCL EXTENSION cl_khr_int64_base_atomics : enable
#pragma OPENCL EXTENSION cl_khr_fp64 : enable

// Kernel 1: accumulate dangling node rank
__kernel void kernel_dangling_sum(
    __global const double* rank,
    __global const int*    out_deg,
    __global double*       dangling_sum,
    int n)
{
    int v = get_global_id(0);
    if (v >= n) return;
    if (out_deg[v] == 0) {
        // atomic_add for double via compare-and-swap loop
        __global ulong* addr = (__global ulong*)dangling_sum;
        ulong old_bits, new_bits;
        double old_val, new_val;
        do {
            old_bits = *addr;
            old_val  = as_double(old_bits);
            new_val  = old_val + rank[v];
            new_bits = as_ulong(new_val);
        } while (atom_cmpxchg(addr, old_bits, new_bits) != old_bits);
    }
}

// Kernel 2: initialise new_rank with teleport + dangling base
__kernel void kernel_init_rank(
    __global double* new_rank,
    double teleport,
    double dangling_contrib,
    int n)
{
    int v = get_global_id(0);
    if (v >= n) return;
    new_rank[v] = teleport + dangling_contrib;
}

// Kernel 3: scatter rank via in-edges
__kernel void kernel_scatter(
    __global const double* rank,
    __global const int*    out_deg,
    __global const int*    in_row_ptr,
    __global const int*    in_dst,
    __global       double* new_rank,
    double damping,
    int n)
{
    int v = get_global_id(0);
    if (v >= n) return;
    double contrib = 0.0;
    int start = in_row_ptr[v];
    int end   = in_row_ptr[v + 1];
    for (int k = start; k < end; k++) {
        int u = in_dst[k];
        contrib += rank[u] / (double)out_deg[u];
    }
    new_rank[v] += damping * contrib;
}

// Kernel 4: compute max absolute difference (IEEE754 trick)
__kernel void kernel_diff(
    __global const double*  rank,
    __global const double*  new_rank,
    __global ulong*         max_diff_bits,
    int n)
{
    int v = get_global_id(0);
    if (v >= n) return;
    double diff = fabs(new_rank[v] - rank[v]);
    ulong bits  = as_ulong(diff);
    // atomic max via CAS loop
    ulong old_bits, new_bits;
    do {
        old_bits = *max_diff_bits;
        if (old_bits >= bits) break;
        new_bits = bits;
    } while (atom_cmpxchg(max_diff_bits, old_bits, new_bits) != old_bits);
}
)CL";

//GRAPH CSR HOST SIDE
struct Graph {
    int              n;
    long             m;
    std::vector<int> out_deg;
    std::vector<int> row_ptr;
    std::vector<int> dst;
    std::vector<int> in_row_ptr;
    std::vector<int> in_dst;
};

void build_csr(Graph& g, std::vector<std::pair<int,int>>& edges) {
    g.m = (long)edges.size();
    g.out_deg.assign(g.n, 0);
    for (auto& [u, v] : edges) g.out_deg[u]++;
    g.row_ptr.resize(g.n + 1, 0);
    for (int i = 0; i < g.n; i++)
        g.row_ptr[i + 1] = g.row_ptr[i] + g.out_deg[i];
    g.dst.resize(g.m);
    std::vector<int> pos(g.row_ptr.begin(), g.row_ptr.end());
    for (auto& [u, v] : edges) g.dst[pos[u]++] = v;
    std::vector<int> in_deg(g.n, 0);
    for (auto& [u, v] : edges) in_deg[v]++;
    g.in_row_ptr.resize(g.n + 1, 0);
    for (int i = 0; i < g.n; i++)
        g.in_row_ptr[i + 1] = g.in_row_ptr[i] + in_deg[i];
    g.in_dst.resize(g.m);
    std::vector<int> ipos(g.in_row_ptr.begin(), g.in_row_ptr.end());
    for (auto& [u, v] : edges) g.in_dst[ipos[v]++] = u;
}

Graph build_random_graph(int n, long target_edges, unsigned seed = 42) {
    Graph g; g.n = n;
    std::mt19937 rng(seed);
    std::uniform_int_distribution<int> dist(0, n - 1);
    std::vector<std::pair<int,int>> edges;
    edges.reserve(target_edges);
    long attempts = 0;
    while ((long)edges.size() < target_edges && attempts < target_edges * 4) {
        int u = dist(rng), v = dist(rng);
        if (u != v) edges.push_back({u, v});
        attempts++;
    }
    build_csr(g, edges);
    return g;
}

Graph load_snap_graph(const std::string& filename) {
    std::ifstream file(filename);
    if (!file.is_open()) {
        std::cerr << "ERROR: Cannot open: " << filename << "\n";
        std::exit(1);
    }
    std::cout << "Reading: " << filename << " ...\n";
    std::vector<std::pair<int,int>> edges;
    std::string line;
    int max_node = 0; long self_loops = 0;
    while (std::getline(file, line)) {
        if (line.empty() || line[0] == '#') continue;
        std::istringstream ss(line);
        int u, v;
        if (!(ss >> u >> v)) continue;
        if (u == v) { self_loops++; continue; }
        edges.push_back({u, v});
        max_node = std::max(max_node, std::max(u, v));
    }
    std::cout << "  Edges loaded      : " << edges.size()  << "\n";
    std::cout << "  Self-loops skipped: " << self_loops    << "\n";
    std::cout << "  Total nodes       : " << max_node + 1  << "\n\n";
    Graph g; g.n = max_node + 1;
    build_csr(g, edges);
    return g;
}


struct CLContext {
    cl_platform_id   platform;
    cl_device_id     device;
    cl_context       ctx;
    cl_command_queue queue;
    cl_program       program;
    cl_kernel        k_dangling;
    cl_kernel        k_init;
    cl_kernel        k_scatter;
    cl_kernel        k_diff;
    std::string      device_name;

    void init() {
        // Get first GPU device
        cl_uint num_platforms;
        CL_CHECK(clGetPlatformIDs(1, &platform, &num_platforms));
        CL_CHECK(clGetDeviceIDs(platform, CL_DEVICE_TYPE_GPU, 1, &device, nullptr));

        // Print device name
        char name[256];
        clGetDeviceInfo(device, CL_DEVICE_NAME, sizeof(name), name, nullptr);
        device_name = name;

        cl_int err;
        ctx   = clCreateContext(nullptr, 1, &device, nullptr, nullptr, &err); CL_CHECK(err);
        queue = clCreateCommandQueue(ctx, device, CL_QUEUE_PROFILING_ENABLE, &err); CL_CHECK(err);

        // Compile kernels
        program = clCreateProgramWithSource(ctx, 1, &KERNEL_SRC, nullptr, &err); CL_CHECK(err);
        err = clBuildProgram(program, 1, &device, "-cl-std=CL1.2", nullptr, nullptr);
        if (err != CL_SUCCESS) {
            size_t log_size;
            clGetProgramBuildInfo(program, device, CL_PROGRAM_BUILD_LOG, 0, nullptr, &log_size);
            std::string log(log_size, ' ');
            clGetProgramBuildInfo(program, device, CL_PROGRAM_BUILD_LOG, log_size, &log[0], nullptr);
            std::cerr << "Kernel build error:\n" << log << "\n";
            std::exit(1);
        }

        k_dangling = clCreateKernel(program, "kernel_dangling_sum", &err); CL_CHECK(err);
        k_init     = clCreateKernel(program, "kernel_init_rank",    &err); CL_CHECK(err);
        k_scatter  = clCreateKernel(program, "kernel_scatter",      &err); CL_CHECK(err);
        k_diff     = clCreateKernel(program, "kernel_diff",         &err); CL_CHECK(err);
    }

    void cleanup() {
        clReleaseKernel(k_dangling); clReleaseKernel(k_init);
        clReleaseKernel(k_scatter);  clReleaseKernel(k_diff);
        clReleaseProgram(program);
        clReleaseCommandQueue(queue);
        clReleaseContext(ctx);
    }
};


struct PRResult {
    std::vector<double> rank;
    int    iterations;
    double max_diff;
    double time_ms;
    double cpu_mem_mb;
    double gpu_mem_mb;
};

PRResult pagerank_opencl(const Graph& g, CLContext& cl) {
    int  n = g.n;
    long m = g.m;
    cl_int err;

    double cpu_mem_mb = (double)(n * sizeof(double) + n * sizeof(int) * 3 + m * sizeof(int)) / (1024.0 * 1024.0);
    double gpu_mem_mb = (double)(n * 2 * sizeof(double) + n * sizeof(int) + (n+1) * sizeof(int) + m * sizeof(int)) / (1024.0 * 1024.0);

    // ── Allocate GPU buffers ─────────────────────────────────
    cl_mem d_rank       = clCreateBuffer(cl.ctx, CL_MEM_READ_WRITE, n * sizeof(double), nullptr, &err); CL_CHECK(err);
    cl_mem d_new_rank   = clCreateBuffer(cl.ctx, CL_MEM_READ_WRITE, n * sizeof(double), nullptr, &err); CL_CHECK(err);
    cl_mem d_out_deg    = clCreateBuffer(cl.ctx, CL_MEM_READ_ONLY,  n * sizeof(int),    nullptr, &err); CL_CHECK(err);
    cl_mem d_in_row_ptr = clCreateBuffer(cl.ctx, CL_MEM_READ_ONLY,  (n+1) * sizeof(int),nullptr, &err); CL_CHECK(err);
    cl_mem d_in_dst     = clCreateBuffer(cl.ctx, CL_MEM_READ_ONLY,  m * sizeof(int),    nullptr, &err); CL_CHECK(err);
    cl_mem d_dangling   = clCreateBuffer(cl.ctx, CL_MEM_READ_WRITE, sizeof(double),     nullptr, &err); CL_CHECK(err);
    cl_mem d_diff_bits  = clCreateBuffer(cl.ctx, CL_MEM_READ_WRITE, sizeof(cl_ulong),   nullptr, &err); CL_CHECK(err);

    // ── Upload graph ─────────────────────────────────────────
    CL_CHECK(clEnqueueWriteBuffer(cl.queue, d_out_deg,    CL_TRUE, 0, n * sizeof(int),     g.out_deg.data(),    0, nullptr, nullptr));
    CL_CHECK(clEnqueueWriteBuffer(cl.queue, d_in_row_ptr, CL_TRUE, 0, (n+1) * sizeof(int), g.in_row_ptr.data(), 0, nullptr, nullptr));
    CL_CHECK(clEnqueueWriteBuffer(cl.queue, d_in_dst,     CL_TRUE, 0, m * sizeof(int),     g.in_dst.data(),     0, nullptr, nullptr));

    // ── Initialise rank = 1/n ────────────────────────────────
    std::vector<double> h_rank(n, 1.0 / n);
    CL_CHECK(clEnqueueWriteBuffer(cl.queue, d_rank, CL_TRUE, 0, n * sizeof(double), h_rank.data(), 0, nullptr, nullptr));

    const double teleport = (1.0 - DAMPING) / n;
    const size_t global_size = ((n + 255) / 256) * 256;
    const size_t local_size  = 256;

    // ── Timing ──────────────────────────────────────────────
    auto t0 = std::chrono::high_resolution_clock::now();

    int    iter = 0;
    double diff = 1.0;

    while (diff > EPSILON && iter < MAX_ITER) {

        // Step 1: dangling sum
        double zero_d = 0.0;
        CL_CHECK(clEnqueueWriteBuffer(cl.queue, d_dangling, CL_TRUE, 0, sizeof(double), &zero_d, 0, nullptr, nullptr));
        CL_CHECK(clSetKernelArg(cl.k_dangling, 0, sizeof(cl_mem), &d_rank));
        CL_CHECK(clSetKernelArg(cl.k_dangling, 1, sizeof(cl_mem), &d_out_deg));
        CL_CHECK(clSetKernelArg(cl.k_dangling, 2, sizeof(cl_mem), &d_dangling));
        CL_CHECK(clSetKernelArg(cl.k_dangling, 3, sizeof(int),    &n));
        CL_CHECK(clEnqueueNDRangeKernel(cl.queue, cl.k_dangling, 1, nullptr, &global_size, &local_size, 0, nullptr, nullptr));

        double h_dangling = 0.0;
        CL_CHECK(clEnqueueReadBuffer(cl.queue, d_dangling, CL_TRUE, 0, sizeof(double), &h_dangling, 0, nullptr, nullptr));
        double dangling_contrib = DAMPING * h_dangling / n;

        // Step 2: init new_rank
        CL_CHECK(clSetKernelArg(cl.k_init, 0, sizeof(cl_mem), &d_new_rank));
        CL_CHECK(clSetKernelArg(cl.k_init, 1, sizeof(double), &teleport));
        CL_CHECK(clSetKernelArg(cl.k_init, 2, sizeof(double), &dangling_contrib));
        CL_CHECK(clSetKernelArg(cl.k_init, 3, sizeof(int),    &n));
        CL_CHECK(clEnqueueNDRangeKernel(cl.queue, cl.k_init, 1, nullptr, &global_size, &local_size, 0, nullptr, nullptr));

        // Step 3: scatter rank
        double damping_d = DAMPING;
        CL_CHECK(clSetKernelArg(cl.k_scatter, 0, sizeof(cl_mem), &d_rank));
        CL_CHECK(clSetKernelArg(cl.k_scatter, 1, sizeof(cl_mem), &d_out_deg));
        CL_CHECK(clSetKernelArg(cl.k_scatter, 2, sizeof(cl_mem), &d_in_row_ptr));
        CL_CHECK(clSetKernelArg(cl.k_scatter, 3, sizeof(cl_mem), &d_in_dst));
        CL_CHECK(clSetKernelArg(cl.k_scatter, 4, sizeof(cl_mem), &d_new_rank));
        CL_CHECK(clSetKernelArg(cl.k_scatter, 5, sizeof(double), &damping_d));
        CL_CHECK(clSetKernelArg(cl.k_scatter, 6, sizeof(int),    &n));
        CL_CHECK(clEnqueueNDRangeKernel(cl.queue, cl.k_scatter, 1, nullptr, &global_size, &local_size, 0, nullptr, nullptr));

        // Step 4: convergence diff
        cl_ulong zero_ull = 0ULL;
        CL_CHECK(clEnqueueWriteBuffer(cl.queue, d_diff_bits, CL_TRUE, 0, sizeof(cl_ulong), &zero_ull, 0, nullptr, nullptr));
        CL_CHECK(clSetKernelArg(cl.k_diff, 0, sizeof(cl_mem), &d_rank));
        CL_CHECK(clSetKernelArg(cl.k_diff, 1, sizeof(cl_mem), &d_new_rank));
        CL_CHECK(clSetKernelArg(cl.k_diff, 2, sizeof(cl_mem), &d_diff_bits));
        CL_CHECK(clSetKernelArg(cl.k_diff, 3, sizeof(int),    &n));
        CL_CHECK(clEnqueueNDRangeKernel(cl.queue, cl.k_diff, 1, nullptr, &global_size, &local_size, 0, nullptr, nullptr));

        cl_ulong h_bits = 0ULL;
        CL_CHECK(clEnqueueReadBuffer(cl.queue, d_diff_bits, CL_TRUE, 0, sizeof(cl_ulong), &h_bits, 0, nullptr, nullptr));
        memcpy(&diff, &h_bits, sizeof(double));

        // Step 5: swap buffers
        std::swap(d_rank, d_new_rank);
        iter++;
    }

    CL_CHECK(clFinish(cl.queue));
    auto t1 = std::chrono::high_resolution_clock::now();
    double ms = std::chrono::duration<double, std::milli>(t1 - t0).count();

    // ── Copy result back ─────────────────────────────────────
    CL_CHECK(clEnqueueReadBuffer(cl.queue, d_rank, CL_TRUE, 0, n * sizeof(double), h_rank.data(), 0, nullptr, nullptr));

    // ── Cleanup ──────────────────────────────────────────────
    clReleaseMemObject(d_rank);     clReleaseMemObject(d_new_rank);
    clReleaseMemObject(d_out_deg);  clReleaseMemObject(d_in_row_ptr);
    clReleaseMemObject(d_in_dst);   clReleaseMemObject(d_dangling);
    clReleaseMemObject(d_diff_bits);

    return { h_rank, iter, diff, ms, cpu_mem_mb, gpu_mem_mb };
}

// ════════════════════════════════════════════════════════════
//  Print results
// ════════════════════════════════════════════════════════════
void print_results(const std::string& label, const Graph& g, const PRResult& r, const std::string& device) {
    double throughput = (double)g.m * r.iterations / r.time_ms;
    int top_k = std::min(5, g.n);
    std::vector<int> idx(g.n);
    std::iota(idx.begin(), idx.end(), 0);
    std::partial_sort(idx.begin(), idx.begin() + top_k, idx.end(),
        [&](int a, int b){ return r.rank[a] > r.rank[b]; });

    std::cout << "Device: " << device << "\n";
    std::cout << "=============================\n";
    std::cout << "  Dataset     : " << label          << "\n";
    std::cout << "  Nodes       : " << g.n            << "\n";
    std::cout << "  Edges       : " << g.m            << "\n";
    std::cout << "  Iterations  : " << r.iterations   << "\n";
    std::cout << "  Converged   : "
              << (r.max_diff < EPSILON ? "Yes" : "No (hit MAX_ITER)")
              << "  (max|diff| = "
              << std::scientific << std::setprecision(2) << r.max_diff << ")\n";
    std::cout << std::fixed << std::setprecision(6);
    std::cout << "  Time        : " << r.time_ms     << " ms\n";
    std::cout << "  GPU Mem     : " << r.gpu_mem_mb  << " MB\n";
    std::cout << "  CPU Mem     : " << r.cpu_mem_mb  << " MB\n";
    std::cout << "  Total Mem   : " << r.gpu_mem_mb + r.cpu_mem_mb << " MB\n";
    std::cout << std::fixed << std::setprecision(0);
    std::cout << "  Throughput  : " << throughput << " edge-iters/ms\n";
    std::cout << "  Top-5 Nodes :\n";
    for (int i = 0; i < top_k; i++) {
        std::cout << "    Rank " << (i+1) << " → Node #"
                  << std::setw(8) << idx[i]
                  << "   PR = " << std::scientific << std::setprecision(6)
                  << r.rank[idx[i]] << "\n";
    }
    std::cout << "=============================\n\n";
}


int main(int argc, char* argv[]) {

    CLContext cl;
    cl.init();

    std::cout << "==============================================\n";
    std::cout << "  OpenCL PageRank Benchmark\n";
    std::cout << "  Device : " << cl.device_name << "\n";
    std::cout << "  d=" << DAMPING << "  eps=" << EPSILON
              << "  max_iter=" << MAX_ITER << "\n";
    std::cout << "==============================================\n\n";

    // ── SNAP file mode ───────────────────────────────────────
    if (argc == 2) {
        std::string filename = argv[1];
        Graph    g = load_snap_graph(filename);
        PRResult r = pagerank_opencl(g, cl);
        std::string label = filename;
        size_t slash = label.find_last_of("/\\");
        if (slash != std::string::npos) label = label.substr(slash + 1);
        print_results(label, g, r, cl.device_name);
        cl.cleanup();
        return 0;
    }

    // ── Synthetic benchmark mode ─────────────────────────────
    std::cout << "--------------------------------------------\n";
    std::cout << "  PART 1 — Synthetic Random Graphs\n";
    std::cout << "--------------------------------------------\n\n";

    struct Dataset { std::string name; int n; long m; };
    std::vector<Dataset> datasets = {
        { "Small",   1000,     7992    },
        { "Medium",  64000,    511993  },
        { "Large",   256000,   2047993 },
        { "XLarge",  1000000,  7999994 },
    };
    for (auto& d : datasets) {
        std::cout << "Building " << d.name << " graph...\n";
        Graph    g = build_random_graph(d.n, d.m);
        PRResult r = pagerank_opencl(g, cl);
        print_results(d.name, g, r, cl.device_name);
    }

    // ── SNAP web-Stanford ────────────────────────────────────
    std::cout << "--------------------------------------------\n";
    std::cout << "  PART 2 — Real-World SNAP Dataset\n";
    std::cout << "--------------------------------------------\n\n";
    Graph    g = load_snap_graph("web-Stanford.txt");
    PRResult r = pagerank_opencl(g, cl);
    print_results("web-Stanford", g, r, cl.device_name);

    cl.cleanup();
    return 0;
}

Writing pagerank_opencl.cpp


In [ ]:
# Install OpenCL headers
!apt-get install -y ocl-icd-opencl-dev

# Compile
!g++ -O2 -std=c++17 -o pagerank_opencl pagerank_opencl.cpp -lOpenCL

# Run
!./pagerank_opencl

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ocl-icd-opencl-dev is already the newest version (2.2.14-3).
ocl-icd-opencl-dev set to manually installed.
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
In file included from /usr/include/CL/cl.h:20,
                 from pagerank_opencl.cpp:15:
/usr/include/CL/cl_version.h:22:104: note: ‘#pragma message: cl_version.h: CL_TARGET_OPENCL_VERSION is not defined. Defaulting to 300 (OpenCL 3.0)’
   22 | #pragma message("cl_version.h: CL_TARGET_OPENCL_VERSION is not defined. Defaulting to 300 (OpenCL 3.0)")
      |                                                                                                        ^
pagerank_opencl.cpp: In member function ‘void CLContext::init()’:
pagerank_opencl.cpp:236:37: warning: ‘_cl_command_queue* clCreateCommandQueue(cl_context, cl_device_id, cl_command_queue_properties, cl_int*)’ is deprecated []8;;https://gcc.gnu.org/onlinedocs/gcc/W

In [ ]:
#SSSP - BELLMANFORD CPU


In [ ]:
%%writefile sssp_cpu.c
/*
 * sssp_cpu_fixed.c  —  Bellman-Ford with memory measurement
 *
 * Compile & run:
 *   gcc -O2 -o bf_cpu sssp_cpu_fixed.c -lm && ./bf_cpu
 *
 * Memory is measured via /proc/self/status (VmRSS) — actual physical
 * RAM pages used by the process, sampled before and after each run.
 */

#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>
#include <math.h>

#define INF 1e18

typedef struct { int u, v; double w; } Edge;

/* ── /proc/self/status memory reader ─────────────────────────────────────
   VmRSS  = current physical RAM used (KB)
   VmPeak = peak virtual memory ever used (KB)                            */
static void get_mem_kb(long *rss_kb, long *peak_kb) {
    *rss_kb = *peak_kb = -1;
    FILE *f = fopen("/proc/self/status", "r");
    if (!f) return;
    char line[128];
    while (fgets(line, sizeof(line), f)) {
        long v;
        if (sscanf(line, "VmRSS: %ld kB",  &v) == 1) *rss_kb  = v;
        if (sscanf(line, "VmPeak: %ld kB", &v) == 1) *peak_kb = v;
    }
    fclose(f);
}

static void print_mem_delta(long rss_before, long rss_after,
                            long peak_after, long E, int V) {
    double used_mb  = (rss_after  - rss_before) / 1024.0;
    double peak_mb  =  peak_after               / 1024.0;
    double theory_mb = ((double)E * sizeof(Edge)
                      + (double)V * (sizeof(double) + sizeof(int)))
                      / (1024.0 * 1024.0);
    printf("Memory     : used=%.1f MB  peak(process)=%.1f MB  "
           "theoretical=%.1f MB\n", used_mb, peak_mb, theory_mb);
}

/* ── Bellman-Ford ─────────────────────────────────────────────────────── */
static int bellman_ford(int V, long E, Edge *edges, int src,
                        double *dist, int *pred)
{
    for (int i = 0; i < V; i++) { dist[i] = INF; pred[i] = -1; }
    dist[src] = 0.0;
    int iters = 0;
    for (int i = 0; i < V - 1; i++) {
        int updated = 0;
        for (long e = 0; e < E; e++) {
            int u = edges[e].u, v = edges[e].v; double w = edges[e].w;
            if (dist[u] < INF && dist[u] + w < dist[v]) {
                dist[v] = dist[u] + w; pred[v] = u; updated = 1;
            }
        }
        iters++;
        if (!updated) break;
    }
    for (long e = 0; e < E; e++) {
        int u = edges[e].u, v = edges[e].v;
        if (dist[u] < INF && dist[u] + edges[e].w < dist[v]) return -1;
    }
    return iters;
}

/* ── Helpers ──────────────────────────────────────────────────────────── */
static double elapsed_ms(struct timespec *a, struct timespec *b) {
    return (b->tv_sec - a->tv_sec)*1000.0 + (b->tv_nsec - a->tv_nsec)/1e6;
}

static unsigned long rng_state = 123456789UL;
static int rand_int(int lo, int hi) {
    rng_state ^= rng_state << 13;
    rng_state ^= rng_state >> 7;
    rng_state ^= rng_state << 17;
    return lo + (int)(rng_state % (unsigned long)(hi - lo));
}

static void print_path(int *pred, int src, int dst) {
    int path[10000], len = 0, cur = dst;
    while (cur != -1 && len < 10000) {
        path[len++] = cur; if (cur == src) break; cur = pred[cur];
    }
    if (!len || path[len-1] != src) { printf("unreachable"); return; }
    int start = (len > 8) ? len-8 : 0;
    if (start > 0) printf("... -> ");
    for (int i = len-1; i >= start; i--) {
        printf("%d", path[i]); if (i > start) printf(" -> ");
    }
}

static void print_stats(int V, long E, double *dist, int iters, double ms) {
    long reach = 0; double sum = 0, dmax = 0;
    for (int v = 1; v < V; v++)
        if (dist[v] < INF) { reach++; sum += dist[v]; if (dist[v]>dmax) dmax=dist[v]; }
    printf("Source     : 0\n");
    printf("Iterations : %d  (max would be %d)\n", iters, V-1);
    if (ms < 1000.0) printf("Time       : %.2f ms\n", ms);
    else             printf("Time       : %.1f ms  (%.2f s)\n", ms, ms/1000.0);
    printf("Throughput : %.0f edges/ms\n", E / ms);
    printf("Neg cycle  : %s\n", iters==-1?"YES":"no");
    printf("Reachable  : %ld / %d  (%.1f%%)\n", reach, V-1, 100.0*reach/(V-1));
    printf("Dist max   : %.0f\n", dmax);
    printf("Dist mean  : %.1f\n", reach ? sum/reach : 0.0);
}

/* ── Small ────────────────────────────────────────────────────────────── */
static void run_small(void) {
    printf("\n╔══════════════════════════════════════════════╗\n");
    printf(  "║  SMALL GRAPH  (V=8, E=11)                   ║\n");
    printf(  "║  Hand-crafted · includes negative edge       ║\n");
    printf(  "╚══════════════════════════════════════════════╝\n");
    int V = 8;
    Edge edges[] = {
        {0,1,4},{0,2,2},{1,3,3},{1,4,5},{2,4,1},
        {3,5,2},{4,3,-2},{4,5,1},{0,6,8},{5,7,3},{6,7,10}
    };
    int E = 11;
    double *dist = malloc(V * sizeof(double));
    int    *pred = malloc(V * sizeof(int));

    long rss_before, rss_after, peak_after, dummy;
    get_mem_kb(&rss_before, &dummy);
    struct timespec t0, t1;
    clock_gettime(CLOCK_MONOTONIC, &t0);
    int iters = bellman_ford(V, E, edges, 0, dist, pred);
    clock_gettime(CLOCK_MONOTONIC, &t1);
    get_mem_kb(&rss_after, &peak_after);

    printf("Source     : 0\n");
    printf("Iterations : %d  (max would be %d)\n", iters, V-1);
    printf("Time       : %.4f ms\n", elapsed_ms(&t0, &t1));
    printf("Neg cycle  : %s\n", iters==-1?"YES":"no");
    print_mem_delta(rss_before, rss_after, peak_after, E, V);
    printf("\n");
    printf("  %-8s  %-10s  %s\n","Vertex","Distance","Path");
    printf("  %-8s  %-10s  %s\n","------","--------","----");
    for (int v = 0; v < V; v++) {
        if (dist[v] >= INF) printf("  %-8d  %-10s  unreachable\n", v, "inf");
        else { printf("  %-8d  %-10.0f  ", v, dist[v]); print_path(pred,0,v); printf("\n"); }
    }
    free(dist); free(pred);
}

/* ── Medium ───────────────────────────────────────────────────────────── */
static void run_medium(void) {
    printf("\n╔══════════════════════════════════════════════╗\n");
    printf(  "║  MEDIUM GRAPH  (V=100,000  E=1,000,000)     ║\n");
    printf(  "║  Random · positive weights [1,50]            ║\n");
    printf(  "╚══════════════════════════════════════════════╝\n");
    int V = 100000; long E_cap = 1000000;
    rng_state = 42UL;

    long rss_before, peak_after, dummy;
    get_mem_kb(&rss_before, &dummy);

    Edge *edges = malloc(E_cap * sizeof(Edge));
    if (!edges) { printf("malloc failed\n"); return; }
    long E = 0;
    while (E < E_cap) {
        int u = rand_int(0,V), v = rand_int(0,V);
        if (u==v) continue;
        edges[E].u=u; edges[E].v=v; edges[E].w=rand_int(1,51); E++;
    }
    double *dist = malloc(V * sizeof(double));
    int    *pred = malloc(V * sizeof(int));

    struct timespec t0, t1;
    clock_gettime(CLOCK_MONOTONIC, &t0);
    int iters = bellman_ford(V, E, edges, 0, dist, pred);
    clock_gettime(CLOCK_MONOTONIC, &t1);

    long rss_after;
    get_mem_kb(&rss_after, &peak_after);

    print_stats(V, E, dist, iters, elapsed_ms(&t0,&t1));
    print_mem_delta(rss_before, rss_after, peak_after, E, V);

    printf("\nFirst 10 reachable vertices:\n");
    printf("  %-8s  %-10s  %s\n","Vertex","Distance","Path (first 8 hops)");
    printf("  %-8s  %-10s  %s\n","------","--------","-------------------");
    int shown = 0;
    for (int v = 1; v < V && shown < 10; v++) {
        if (dist[v] < INF) {
            printf("  %-8d  %-10.0f  ", v, dist[v]);
            print_path(pred, 0, v); printf("\n"); shown++;
        }
    }
    free(edges); free(dist); free(pred);
}

/* ── Large ────────────────────────────────────────────────────────────── */
static void run_large(void) {
    printf("\n╔══════════════════════════════════════════════╗\n");
    printf(  "║  LARGE GRAPH  (V=1,000,000  E=10,000,000)   ║\n");
    printf(  "║  Random · positive weights [1,100]           ║\n");
    printf(  "╚══════════════════════════════════════════════╝\n");
    int V = 1000000; long E_cap = 10000000;
    rng_state = 99UL;

    long rss_before, rss_after, peak_after, dummy;
    get_mem_kb(&rss_before, &dummy);

    printf("Allocating %.0f MB for edges...\n", E_cap*sizeof(Edge)/1048576.0);
    fflush(stdout);
    Edge *edges = malloc(E_cap * sizeof(Edge));
    if (!edges) { printf("malloc failed\n"); return; }
    long E = 0;
    while (E < E_cap) {
        int u = rand_int(0,V), v = rand_int(0,V);
        if (u==v) continue;
        edges[E].u=u; edges[E].v=v; edges[E].w=rand_int(1,101); E++;
    }
    double *dist = malloc(V * sizeof(double));
    int    *pred = malloc(V * sizeof(int));

    printf("Graph built. Running Bellman-Ford...\n"); fflush(stdout);
    struct timespec t0, t1;
    clock_gettime(CLOCK_MONOTONIC, &t0);
    int iters = bellman_ford(V, E, edges, 0, dist, pred);
    clock_gettime(CLOCK_MONOTONIC, &t1);
    get_mem_kb(&rss_after, &peak_after);

    print_stats(V, E, dist, iters, elapsed_ms(&t0,&t1));
    print_mem_delta(rss_before, rss_after, peak_after, E, V);
    free(edges); free(dist); free(pred);
}

/* ── XLarge ───────────────────────────────────────────────────────────── */
static void run_xlarge(void) {
    printf("\n╔══════════════════════════════════════════════╗\n");
    printf(  "║  EXTRA-LARGE GRAPH  (V=10M  E=50M)          ║\n");
    printf(  "║  Random · positive weights [1,100]           ║\n");
    printf(  "╚══════════════════════════════════════════════╝\n");
    int V = 10000000; long E_cap = 50000000;
    rng_state = 777UL;

    long rss_before, rss_after, peak_after, dummy;
    get_mem_kb(&rss_before, &dummy);

    printf("Allocating %.0f MB for edges...\n", E_cap*sizeof(Edge)/1048576.0);
    fflush(stdout);
    Edge *edges = malloc(E_cap * sizeof(Edge));
    if (!edges) { printf("malloc failed — not enough RAM\n"); return; }
    long E = 0;
    while (E < E_cap) {
        int u = rand_int(0,V), v = rand_int(0,V);
        if (u==v) continue;
        edges[E].u=u; edges[E].v=v; edges[E].w=rand_int(1,101); E++;
    }
    double *dist = malloc((size_t)V * sizeof(double));
    int    *pred = malloc((size_t)V * sizeof(int));
    if (!dist || !pred) { printf("malloc failed\n"); free(edges); return; }

    printf("Graph built. Running Bellman-Ford...\n"); fflush(stdout);
    struct timespec t0, t1;
    clock_gettime(CLOCK_MONOTONIC, &t0);
    int iters = bellman_ford(V, E, edges, 0, dist, pred);
    clock_gettime(CLOCK_MONOTONIC, &t1);
    get_mem_kb(&rss_after, &peak_after);

    print_stats(V, E, dist, iters, elapsed_ms(&t0,&t1));
    print_mem_delta(rss_before, rss_after, peak_after, E, V);
    free(edges); free(dist); free(pred);
}

/* ── Negative cycle demo ──────────────────────────────────────────────── */
static void run_neg_cycle(void) {
    printf("\n╔══════════════════════════════════════════════╗\n");
    printf(  "║  NEGATIVE CYCLE DEMO                        ║\n");
    printf(  "║  Cycle 1->2->3->1  cost = 2 + (-4) + 1 = -1║\n");
    printf(  "╚══════════════════════════════════════════════╝\n");
    int V = 5;
    Edge edges[] = {{0,1,1},{1,2,2},{2,3,-4},{3,1,1},{0,4,5},{3,4,2}};
    int E = 6;
    double *dist = malloc(V * sizeof(double));
    int    *pred = malloc(V * sizeof(int));
    int iters = bellman_ford(V, E, edges, 0, dist, pred);
    printf("Neg cycle detected : %s\n", iters==-1?"YES ✓":"no");
    printf("(distances are unreliable when a neg cycle exists)\n");
    free(dist); free(pred);
}

/* ── Main ─────────────────────────────────────────────────────────────── */
int main(void) {
    printf("=======================================================\n");
    printf("  Bellman-Ford SSSP — Small / Medium / Large / XLarge \n");
    printf("  Graph sizes matched to CUDA & OpenCL versions        \n");
    printf("  Memory measured via /proc/self/status (VmRSS)        \n");
    printf("=======================================================\n");
    run_small();
    run_medium();
    run_large();
    run_xlarge();
    run_neg_cycle();
    printf("\n=======================================================\n");
    printf("  Done.\n");
    printf("=======================================================\n");
    return 0;
}

Writing sssp_cpu.c


In [ ]:
!gcc -O2 -o bf sssp_cpu.c && ./bf

  Bellman-Ford SSSP — Small / Medium / Large / XLarge 
  Graph sizes matched to CUDA & OpenCL versions        
  Memory measured via /proc/self/status (VmRSS)        

╔══════════════════════════════════════════════╗
║  SMALL GRAPH  (V=8, E=11)                   ║
║  Hand-crafted · includes negative edge       ║
╚══════════════════════════════════════════════╝
Source     : 0
Iterations : 3  (max would be 7)
Time       : 0.0004 ms
Neg cycle  : no
Memory     : used=0.2 MB  peak(process)=2.7 MB  theoretical=0.0 MB

  Vertex    Distance    Path
  ------    --------    ----
  0         0           0
  1         4           0 -> 1
  2         2           0 -> 2
  3         1           0 -> 2 -> 4 -> 3
  4         3           0 -> 2 -> 4
  5         3           0 -> 2 -> 4 -> 3 -> 5
  6         8           0 -> 6
  7         6           0 -> 2 -> 4 -> 3 -> 5 -> 7

╔══════════════════════════════════════════════╗
║  MEDIUM GRAPH  (V=100,000  E=1,000,000)     ║
║  Random · positive weights [1,5

In [ ]:
%%writefile sssp_cuda.cu
/*
 * sssp_cuda_fixed.cu  —  Bellman-Ford with memory measurement
 *
 * Compile & run on Google Colab:
 *   !nvcc -O2 -o bf_cuda sssp_cuda_fixed.cu && ./bf_cuda
 *
 * Memory reporting:
 *   - GPU: cudaMemGetInfo() sampled before/after each buffer allocation
 *   - CPU: /proc/self/status VmRSS sampled before/after each run
 *   - Reported as: CPU used | GPU used | total
 */

#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <time.h>

#define INF_F 1e18f

/* ── /proc/self/status RSS reader (CPU RAM) ──────────────────────────── */
static long get_rss_kb(void) {
    FILE *f = fopen("/proc/self/status", "r");
    if (!f) return -1;
    char line[128]; long kb = -1;
    while (fgets(line, sizeof(line), f))
        if (sscanf(line, "VmRSS: %ld kB", &kb) == 1) break;
    fclose(f);
    return kb;
}

/* ── GPU free memory via cudaMemGetInfo ───────────────────────────────── */
static size_t gpu_free_bytes(void) {
    size_t free_b, total_b;
    cudaMemGetInfo(&free_b, &total_b);
    return free_b;
}

static void print_mem(long cpu_before_kb, long cpu_after_kb,
                      size_t gpu_free_before, size_t gpu_free_after) {
    double cpu_mb = (cpu_after_kb - cpu_before_kb) / 1024.0;
    double gpu_mb = (double)(gpu_free_before - gpu_free_after) / (1024.0*1024.0);
    printf("Memory     : CPU=%.1f MB  GPU=%.1f MB  total=%.1f MB\n",
           cpu_mb, gpu_mb, cpu_mb + gpu_mb);
}

/* ── Timing helper ────────────────────────────────────────────────────── */
static double elapsed_ms(struct timespec *a, struct timespec *b) {
    return (b->tv_sec - a->tv_sec)*1000.0 + (b->tv_nsec - a->tv_nsec)/1e6;
}

/* ── xorshift RNG ─────────────────────────────────────────────────────── */
static unsigned long rng = 123456789UL;
static int randi(int lo, int hi) {
    rng ^= rng<<13; rng ^= rng>>7; rng ^= rng<<17;
    return lo + (int)(rng % (unsigned long)(hi-lo));
}

/* ── Build random edge list ───────────────────────────────────────────── */
static void build(int V, int Ecap,
                  int **s, int **d, float **w, int *E) {
    *s = (int  *)malloc((size_t)Ecap * sizeof(int));
    *d = (int  *)malloc((size_t)Ecap * sizeof(int));
    *w = (float*)malloc((size_t)Ecap * sizeof(float));
    if (!*s || !*d || !*w) { fprintf(stderr,"malloc failed\n"); exit(1); }
    int n = 0;
    while (n < Ecap) {
        int u = randi(0,V), v = randi(0,V);
        if (u==v) continue;
        (*s)[n]=u; (*d)[n]=v; (*w)[n]=(float)randi(1,101); n++;
    }
    *E = n;
}

/* ── Predecessor reconstruction (CPU pass after GPU run) ─────────────── */
static void make_pred(int V, int E,
                      int *src, int *dst, float *wgt,
                      float *dist, int *pred) {
    for (int i=0; i<V; i++) pred[i]=-1;
    for (int e=0; e<E; e++) {
        int u=src[e], v=dst[e];
        if (dist[u]<INF_F*0.5f && fabsf(dist[u]+wgt[e]-dist[v])<0.5f)
            pred[v]=u;
    }
}

static void print_path(int *pred, int src, int v) {
    int path[512], len=0, cur=v;
    while (cur!=-1 && len<512) {
        path[len++]=cur; if (cur==src) break; cur=pred[cur];
    }
    if (!len || path[len-1]!=src) { printf("unreachable"); return; }
    int start=(len>8)?len-8:0;
    if (start>0) printf("... -> ");
    for (int i=len-1; i>=start; i--) {
        printf("%d",path[i]); if (i>start) printf(" -> ");
    }
}

static void print_stats(int V, int E, float *dist, int iters, double ms) {
    long reach=0; double sum=0, dmax=0;
    for (int v=1; v<V; v++)
        if (dist[v]<INF_F*0.5f){reach++;sum+=dist[v];if(dist[v]>dmax)dmax=dist[v];}
    printf("Source     : 0\n");
    printf("Iterations : %d  (max would be %d)\n", iters, V-1);
    if (ms<1000.0) printf("Time       : %.2f ms\n", ms);
    else           printf("Time       : %.1f ms  (%.2f s)\n", ms, ms/1000.0);
    printf("Throughput : %.0f edges/ms\n", E / ms);
    printf("Neg cycle  : %s\n", iters==-1?"YES":"no");
    printf("Reachable  : %ld / %d  (%.1f%%)\n", reach,V-1,100.0*reach/(V-1));
    printf("Dist max   : %.0f\n", dmax);
    printf("Dist mean  : %.1f\n", reach?sum/reach:0.0);
}

/* ── CUDA kernel: one thread per edge ────────────────────────────────── */
__global__ void bf_kernel(const int   *src,
                          const int   *dst,
                          const float *wgt,
                          float       *dist,
                          int         *updated,
                          int          E)
{
    int e = blockIdx.x * blockDim.x + threadIdx.x;
    if (e >= E) return;
    int   u  = src[e], v = dst[e];
    float nd = dist[u] + wgt[e];
    if (dist[u] < 5e17f && nd < dist[v]) {
        atomicExch(&dist[v], fminf(dist[v], nd));   /* best-effort min */
        atomicOr(updated, 1);
    }
}

/* ── Host Bellman-Ford driver ─────────────────────────────────────────── */
static int bf_cuda(int V, int E,
                   int *hs, int *hd, float *hw,
                   int source, float *hdist,
                   double *ms_out,
                   size_t *gpu_free_before_out,
                   size_t *gpu_free_after_out)
{
    for (int i=0; i<V; i++) hdist[i]=INF_F;
    hdist[source]=0.0f;

    *gpu_free_before_out = gpu_free_bytes();

    int   *ds, *dd; float *dw, *ddist; int *dupd;
    cudaMalloc(&ds,    (size_t)E*sizeof(int));
    cudaMalloc(&dd,    (size_t)E*sizeof(int));
    cudaMalloc(&dw,    (size_t)E*sizeof(float));
    cudaMalloc(&ddist, (size_t)V*sizeof(float));
    cudaMalloc(&dupd,  sizeof(int));

    cudaMemcpy(ds,    hs,    (size_t)E*sizeof(int),   cudaMemcpyHostToDevice);
    cudaMemcpy(dd,    hd,    (size_t)E*sizeof(int),   cudaMemcpyHostToDevice);
    cudaMemcpy(dw,    hw,    (size_t)E*sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(ddist, hdist, (size_t)V*sizeof(float), cudaMemcpyHostToDevice);

    *gpu_free_after_out = gpu_free_bytes();   /* after all buffers allocated */

    int threads = 256;
    int blocks  = (E + threads - 1) / threads;

    struct timespec t0, t1;
    clock_gettime(CLOCK_MONOTONIC, &t0);

    int iters=0, upd;
    for (int i=0; i<V-1; i++) {
        upd=0;
        cudaMemcpy(dupd, &upd, sizeof(int), cudaMemcpyHostToDevice);
        bf_kernel<<<blocks,threads>>>(ds,dd,dw,ddist,dupd,E);
        cudaMemcpy(&upd, dupd, sizeof(int), cudaMemcpyDeviceToHost);
        iters++;
        if (!upd) break;
    }
    /* negative cycle check */
    upd=0;
    cudaMemcpy(dupd, &upd, sizeof(int), cudaMemcpyHostToDevice);
    bf_kernel<<<blocks,threads>>>(ds,dd,dw,ddist,dupd,E);
    cudaMemcpy(&upd, dupd, sizeof(int), cudaMemcpyDeviceToHost);
    if (upd) iters=-1;

    cudaMemcpy(hdist, ddist, (size_t)V*sizeof(float), cudaMemcpyDeviceToHost);
    clock_gettime(CLOCK_MONOTONIC, &t1);
    *ms_out = elapsed_ms(&t0, &t1);

    cudaFree(ds); cudaFree(dd); cudaFree(dw); cudaFree(ddist); cudaFree(dupd);
    return iters;
}

/* ── Small ────────────────────────────────────────────────────────────── */
static void run_small(void) {
    printf("\n╔══════════════════════════════════════════════╗\n");
    printf(  "║  SMALL GRAPH  (V=8, E=11)                   ║\n");
    printf(  "║  Hand-crafted · includes negative edge       ║\n");
    printf(  "╚══════════════════════════════════════════════╝\n");
    int V=8;
    int   s[]={0,0,1,1,2,3,4,4,0,5,6};
    int   d[]={1,2,3,4,4,5,3,5,6,7,7};
    float w[]={4,2,3,5,1,2,-2,1,8,3,10};
    int E=11;
    float *dist=(float*)malloc(V*sizeof(float));
    int   *pred=(int  *)malloc(V*sizeof(int));
    double ms; size_t gfb, gfa;
    long cpu_b = get_rss_kb();
    int iters = bf_cuda(V,E,s,d,w,0,dist,&ms,&gfb,&gfa);
    long cpu_a = get_rss_kb();
    make_pred(V,E,s,d,w,dist,pred);
    printf("Source     : 0\n");
    printf("Iterations : %d  (max would be %d)\n", iters, V-1);
    printf("Time       : %.4f ms\n", ms);
    printf("Neg cycle  : %s\n", iters==-1?"YES":"no");
    print_mem(cpu_b, cpu_a, gfb, gfa);
    printf("\n  %-8s  %-10s  %s\n","Vertex","Distance","Path");
    printf(    "  %-8s  %-10s  %s\n","------","--------","----");
    for (int v=0; v<V; v++) {
        if (dist[v]>=INF_F*0.5f) printf("  %-8d  %-10s  unreachable\n",v,"inf");
        else { printf("  %-8d  %-10.0f  ",v,dist[v]); print_path(pred,0,v); printf("\n"); }
    }
    free(dist); free(pred);
}

/* ── Medium ───────────────────────────────────────────────────────────── */
static void run_medium(void) {
    printf("\n╔══════════════════════════════════════════════╗\n");
    printf(  "║  MEDIUM GRAPH  (V=100,000  E=1,000,000)     ║\n");
    printf(  "║  Random · positive weights [1,50]            ║\n");
    printf(  "╚══════════════════════════════════════════════╝\n");
    int V=100000, *s,*d,E; float *w; rng=42UL;
    build(V,1000000,&s,&d,&w,&E);
    float *dist=(float*)malloc((size_t)V*sizeof(float));
    int   *pred=(int  *)malloc((size_t)V*sizeof(int));
    double ms; size_t gfb, gfa;
    long cpu_b = get_rss_kb();
    int iters = bf_cuda(V,E,s,d,w,0,dist,&ms,&gfb,&gfa);
    long cpu_a = get_rss_kb();
    make_pred(V,E,s,d,w,dist,pred);
    print_stats(V,E,dist,iters,ms);
    print_mem(cpu_b, cpu_a, gfb, gfa);
    printf("\nFirst 10 reachable vertices:\n");
    printf("  %-8s  %-10s  %s\n","Vertex","Distance","Path (first 8 hops)");
    printf("  %-8s  %-10s  %s\n","------","--------","-------------------");
    int shown=0;
    for (int v=1; v<V&&shown<10; v++) {
        if (dist[v]<INF_F*0.5f) {
            printf("  %-8d  %-10.0f  ",v,dist[v]);
            print_path(pred,0,v); printf("\n"); shown++;
        }
    }
    free(s);free(d);free(w);free(dist);free(pred);
}

/* ── Large ────────────────────────────────────────────────────────────── */
static void run_large(void) {
    printf("\n╔══════════════════════════════════════════════╗\n");
    printf(  "║  LARGE GRAPH  (V=1,000,000  E=10,000,000)   ║\n");
    printf(  "║  Random · positive weights [1,100]           ║\n");
    printf(  "╚══════════════════════════════════════════════╝\n");
    int V=1000000, *s,*d,E; float *w; rng=99UL;
    printf("Allocating %.0f MB for edges...\n",
           10000000.0*(sizeof(int)*2+sizeof(float))/(1024*1024));
    fflush(stdout);
    build(V,10000000,&s,&d,&w,&E);
    printf("Graph built. Running Bellman-Ford...\n"); fflush(stdout);
    float *dist=(float*)malloc((size_t)V*sizeof(float));
    double ms; size_t gfb, gfa;
    long cpu_b = get_rss_kb();
    int iters = bf_cuda(V,E,s,d,w,0,dist,&ms,&gfb,&gfa);
    long cpu_a = get_rss_kb();
    print_stats(V,E,dist,iters,ms);
    print_mem(cpu_b, cpu_a, gfb, gfa);
    free(s);free(d);free(w);free(dist);
}

/* ── XLarge ───────────────────────────────────────────────────────────── */
static void run_xlarge(void) {
    printf("\n╔══════════════════════════════════════════════╗\n");
    printf(  "║  EXTRA-LARGE GRAPH  (V=10M  E=50M)          ║\n");
    printf(  "║  Random · positive weights [1,100]           ║\n");
    printf(  "╚══════════════════════════════════════════════╝\n");
    int V=10000000, *s,*d,E; float *w; rng=777UL;
    printf("Allocating %.0f MB for edges...\n",
           50000000.0*(sizeof(int)*2+sizeof(float))/(1024*1024));
    fflush(stdout);
    build(V,50000000,&s,&d,&w,&E);
    printf("Graph built. Running Bellman-Ford...\n"); fflush(stdout);
    float *dist=(float*)malloc((size_t)V*sizeof(float));
    if (!dist) { fprintf(stderr,"malloc failed\n"); return; }
    double ms; size_t gfb, gfa;
    long cpu_b = get_rss_kb();
    int iters = bf_cuda(V,E,s,d,w,0,dist,&ms,&gfb,&gfa);
    long cpu_a = get_rss_kb();
    print_stats(V,E,dist,iters,ms);
    print_mem(cpu_b, cpu_a, gfb, gfa);
    free(s);free(d);free(w);free(dist);
}

/* ── Negative cycle demo ──────────────────────────────────────────────── */
static void run_neg(void) {
    printf("\n╔══════════════════════════════════════════════╗\n");
    printf(  "║  NEGATIVE CYCLE DEMO                        ║\n");
    printf(  "║  Cycle 1->2->3->1  cost = 2 + (-4) + 1 = -1║\n");
    printf(  "╚══════════════════════════════════════════════╝\n");
    int V=5;
    int   s[]={0,1,2,3,0,3};
    int   d[]={1,2,3,1,4,4};
    float w[]={1,2,-4,1,5,2};
    int E=6;
    float *dist=(float*)malloc(V*sizeof(float));
    double ms; size_t gfb, gfa;
    int iters=bf_cuda(V,E,s,d,w,0,dist,&ms,&gfb,&gfa);
    printf("Neg cycle detected : %s\n", iters==-1?"YES ✓":"no");
    printf("(distances are unreliable when a neg cycle exists)\n");
    free(dist);
}

/* ── Main ─────────────────────────────────────────────────────────────── */
int main(void) {
    /* Print GPU info */
    int dev=0; cudaDeviceProp p;
    cudaGetDeviceProperties(&p, dev);
    size_t free_b, total_b; cudaMemGetInfo(&free_b, &total_b);
    printf("=======================================================\n");
    printf("  Bellman-Ford SSSP — Small / Medium / Large / XLarge \n");
    printf("  GPU : %s\n", p.name);
    printf("  VRAM: %.0f MB   SMs: %d   CUDA cores: ~%d\n",
           total_b/1048576.0, p.multiProcessorCount,
           p.multiProcessorCount * 128);
    printf("  Memory measured: CPU via /proc/self/status, GPU via cudaMemGetInfo\n");
    printf("=======================================================\n");

    run_small();
    run_medium();
    run_large();
    run_xlarge();
    run_neg();

    printf("\n=======================================================\n");
    printf("  Done.\n");
    printf("=======================================================\n");
    return 0;
}


Overwriting sssp_cuda.cu


In [ ]:
!nvcc -O2 -o bf_cuda sssp_cuda.cu && ./bf_cuda

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
  Bellman-Ford SSSP — Small / Medium / Large / XLarge 
  GPU : NVIDIA A100-SXM4-40GB
  VRAM: 40441 MB   SMs: 108   CUDA cores: ~13824
  Memory measured: CPU via /proc/self/status, GPU via cudaMemGetInfo

╔══════════════════════════════════════════════╗
║  SMALL GRAPH  (V=8, E=11)                   ║
║  Hand-crafted · includes negative edge       ║
╚══════════════════════════════════════════════╝
Source     : 0
Iterations : 6  (max would be 7)
Time       : 14.7377 ms
Neg cycle  : no
Memory     : CPU=24.1 MB  GPU=2.0 MB  total=26.1 MB

  Vertex    Distance    Path
  ------    --------    ----
  0         0           0
  1         4           0 -> 1
  2         2           0 -> 2
  3         1           0 -> 2 -> 4 -> 3
  4         3           0 -> 2 -> 4
  5         3           0 -> 2 -> 4 -> 3 -> 5
  6 

In [ ]:
%%writefile sssp_opencl.c
/*
 * sssp_opencl_fixed.c  —  Bellman-Ford with memory measurement
 *
 * Compile & run on Google Colab:
 *   !apt-get install -y ocl-icd-opencl-dev 2>/dev/null | tail -2
 *   !gcc -O3 -o bf_ocl sssp_opencl_fixed.c -lOpenCL -lm && ./bf_ocl
 *
 * Memory reporting:
 *   - GPU: clGetMemObjectInfo(CL_MEM_SIZE) summed across all live buffers
 *   - CPU: /proc/self/status VmRSS sampled before/after each run
 *   - Reported as: CPU used | GPU used | total
 */

#ifdef __APPLE__
  #include <OpenCL/opencl.h>
#else
  #include <CL/cl.h>
#endif

#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <time.h>

#define INF 1e18f

/* ── /proc/self/status RSS reader ─────────────────────────────────────── */
static long get_rss_kb(void) {
    FILE *f = fopen("/proc/self/status", "r");
    if (!f) return -1;
    char line[128]; long kb = -1;
    while (fgets(line, sizeof(line), f))
        if (sscanf(line, "VmRSS: %ld kB", &kb) == 1) break;
    fclose(f);
    return kb;
}

/* ── Sum GPU buffer sizes via clGetMemObjectInfo ──────────────────────── */
static double sum_gpu_mb(cl_mem *bufs, int n) {
    double total = 0;
    for (int i = 0; i < n; i++) {
        size_t sz = 0;
        clGetMemObjectInfo(bufs[i], CL_MEM_SIZE, sizeof(sz), &sz, NULL);
        total += sz;
    }
    return total / (1024.0 * 1024.0);
}

static void print_mem(long cpu_before_kb, long cpu_after_kb, double gpu_mb) {
    double cpu_mb = (cpu_after_kb - cpu_before_kb) / 1024.0;
    printf("Memory     : CPU=%.1f MB  GPU=%.1f MB  total=%.1f MB\n",
           cpu_mb, gpu_mb, cpu_mb + gpu_mb);
}

/* ── Timing ───────────────────────────────────────────────────────────── */
static double elapsed_ms(struct timespec *a, struct timespec *b) {
    return (b->tv_sec - a->tv_sec)*1000.0 + (b->tv_nsec - a->tv_nsec)/1e6;
}

/* ── xorshift RNG ─────────────────────────────────────────────────────── */
static unsigned long rng = 123456789UL;
static int randi(int lo, int hi) {
    rng ^= rng<<13; rng ^= rng>>7; rng ^= rng<<17;
    return lo + (int)(rng % (unsigned long)(hi-lo));
}

/* ── Build random edge list ───────────────────────────────────────────── */
static void build(int V, int Ecap,
                  int **s, int **d, float **w, int *E) {
    *s=(int  *)malloc((size_t)Ecap*sizeof(int));
    *d=(int  *)malloc((size_t)Ecap*sizeof(int));
    *w=(float*)malloc((size_t)Ecap*sizeof(float));
    if (!*s||!*d||!*w){fprintf(stderr,"malloc failed\n");exit(1);}
    int n=0;
    while(n<Ecap){
        int u=randi(0,V),v=randi(0,V);
        if(u==v)continue;
        (*s)[n]=u;(*d)[n]=v;(*w)[n]=(float)randi(1,101);n++;
    }
    *E=n;
}

/* ── Predecessor reconstruction ──────────────────────────────────────── */
static void make_pred(int V, int E,
                      int *src, int *dst, float *wgt,
                      float *dist, int *pred) {
    for(int i=0;i<V;i++) pred[i]=-1;
    for(int e=0;e<E;e++){
        int u=src[e],v=dst[e];
        if(dist[u]<INF*0.5f && fabsf(dist[u]+wgt[e]-dist[v])<0.5f)
            pred[v]=u;
    }
}

static void print_path(int *pred, int src, int v) {
    int path[512],len=0,cur=v;
    while(cur!=-1&&len<512){path[len++]=cur;if(cur==src)break;cur=pred[cur];}
    if(!len||path[len-1]!=src){printf("unreachable");return;}
    int start=(len>8)?len-8:0;
    if(start>0)printf("... -> ");
    for(int i=len-1;i>=start;i--){printf("%d",path[i]);if(i>start)printf(" -> ");}
}

static void print_stats(int V, int E, float *dist, int iters, double ms) {
    long reach=0; double sum=0,dmax=0;
    for(int v=1;v<V;v++)
        if(dist[v]<INF*0.5f){reach++;sum+=dist[v];if(dist[v]>dmax)dmax=dist[v];}
    printf("Source     : 0\n");
    printf("Iterations : %d  (max would be %d)\n",iters,V-1);
    if(ms<1000.0) printf("Time       : %.2f ms\n",ms);
    else          printf("Time       : %.1f ms  (%.2f s)\n",ms,ms/1000.0);
    printf("Throughput : %.0f edges/ms\n", E/ms);
    printf("Neg cycle  : %s\n",iters==-1?"YES":"no");
    printf("Reachable  : %ld / %d  (%.1f%%)\n",reach,V-1,100.0*reach/(V-1));
    printf("Dist max   : %.0f\n",dmax);
    printf("Dist mean  : %.1f\n",reach?sum/reach:0.0);
}

/* ── OpenCL kernel ────────────────────────────────────────────────────── */
static const char *kernel_src =
"#pragma OPENCL EXTENSION cl_khr_global_int32_base_atomics : enable\n"
"void atomicMinFloat(__global float *addr, float val) {\n"
"    union { float f; int i; } old_u, assumed_u, new_u;\n"
"    old_u.f = *addr;\n"
"    do {\n"
"        assumed_u.i = old_u.i;\n"
"        new_u.f     = fmin(assumed_u.f, val);\n"
"        old_u.i     = atomic_cmpxchg((__global int *)addr, assumed_u.i, new_u.i);\n"
"    } while (assumed_u.i != old_u.i);\n"
"}\n"
"__kernel void bf_kernel(\n"
"    __global const int   *src,\n"
"    __global const int   *dst,\n"
"    __global const float *wgt,\n"
"    __global       float *dist,\n"
"    __global       int   *updated,\n"
"    const          int    E)\n"
"{\n"
"    int e = get_global_id(0);\n"
"    if (e >= E) return;\n"
"    int   u  = src[e], v = dst[e];\n"
"    float nd = dist[u] + wgt[e];\n"
"    if (dist[u] < 5e17f && nd < dist[v]) {\n"
"        atomicMinFloat(dist + v, nd);\n"
"        atomic_or(updated, 1);\n"
"    }\n"
"}\n";

/* ── OpenCL handles (init once) ───────────────────────────────────────── */
static cl_context       ctx;
static cl_command_queue queue;
static cl_program       prog;
static cl_kernel        kern;

static void ocl_init(void) {
    cl_int err;
    cl_platform_id plat; cl_device_id dev;
    clGetPlatformIDs(1, &plat, NULL);
    err = clGetDeviceIDs(plat, CL_DEVICE_TYPE_GPU, 1, &dev, NULL);
    if (err != CL_SUCCESS)
        clGetDeviceIDs(plat, CL_DEVICE_TYPE_CPU, 1, &dev, NULL);

    char name[128]; cl_ulong mem; cl_uint cu;
    clGetDeviceInfo(dev, CL_DEVICE_NAME,              sizeof(name), name, NULL);
    clGetDeviceInfo(dev, CL_DEVICE_GLOBAL_MEM_SIZE,   sizeof(mem),  &mem, NULL);
    clGetDeviceInfo(dev, CL_DEVICE_MAX_COMPUTE_UNITS, sizeof(cu),   &cu,  NULL);
    printf("  OpenCL device : %s\n", name);
    printf("  VRAM          : %.0f MB   CUs: %u\n", mem/1048576.0, cu);
    printf("  Memory measured: CPU via /proc/self/status, GPU via clGetMemObjectInfo\n\n");

    ctx   = clCreateContext(NULL,1,&dev,NULL,NULL,&err);
    cl_queue_properties qp[]={0};
    queue = clCreateCommandQueueWithProperties(ctx,dev,qp,&err);
    prog  = clCreateProgramWithSource(ctx,1,&kernel_src,NULL,&err);
    err   = clBuildProgram(prog,1,&dev,"-cl-fast-relaxed-math",NULL,NULL);
    if (err!=CL_SUCCESS){
        size_t sz; clGetProgramBuildInfo(prog,dev,CL_PROGRAM_BUILD_LOG,0,NULL,&sz);
        char *log=(char*)malloc(sz);
        clGetProgramBuildInfo(prog,dev,CL_PROGRAM_BUILD_LOG,sz,log,NULL);
        fprintf(stderr,"Build error:\n%s\n",log);free(log);exit(1);
    }
    kern = clCreateKernel(prog,"bf_kernel",&err);
}

static void ocl_cleanup(void) {
    clReleaseKernel(kern); clReleaseProgram(prog);
    clReleaseCommandQueue(queue); clReleaseContext(ctx);
}

/* ── Host Bellman-Ford driver ─────────────────────────────────────────── */
static int bf_ocl(int V, int E,
                  int *hs, int *hd, float *hw,
                  int source, float *hdist,
                  double *ms_out, double *gpu_mb_out,
                  long *cpu_before_out, long *cpu_after_out)
{
    cl_int err;
    for(int i=0;i<V;i++) hdist[i]=INF;
    hdist[source]=0.0f;

    *cpu_before_out = get_rss_kb();

    cl_mem ds    = clCreateBuffer(ctx,CL_MEM_READ_ONLY |CL_MEM_COPY_HOST_PTR,(size_t)E*sizeof(int),  hs,&err);
    cl_mem dd    = clCreateBuffer(ctx,CL_MEM_READ_ONLY |CL_MEM_COPY_HOST_PTR,(size_t)E*sizeof(int),  hd,&err);
    cl_mem dw    = clCreateBuffer(ctx,CL_MEM_READ_ONLY |CL_MEM_COPY_HOST_PTR,(size_t)E*sizeof(float),hw,&err);
    cl_mem ddist = clCreateBuffer(ctx,CL_MEM_READ_WRITE|CL_MEM_COPY_HOST_PTR,(size_t)V*sizeof(float),hdist,&err);
    cl_mem dupd  = clCreateBuffer(ctx,CL_MEM_READ_WRITE,sizeof(int),NULL,&err);

    /* measure GPU memory: sum all 5 buffer sizes */
    cl_mem bufs[5]={ds,dd,dw,ddist,dupd};
    *gpu_mb_out = sum_gpu_mb(bufs, 5);

    cl_int Ecl=(cl_int)E;
    clSetKernelArg(kern,0,sizeof(cl_mem),&ds);
    clSetKernelArg(kern,1,sizeof(cl_mem),&dd);
    clSetKernelArg(kern,2,sizeof(cl_mem),&dw);
    clSetKernelArg(kern,3,sizeof(cl_mem),&ddist);
    clSetKernelArg(kern,4,sizeof(cl_mem),&dupd);
    clSetKernelArg(kern,5,sizeof(cl_int),&Ecl);

    size_t local=1024, global=((size_t)E+local-1)/local*local;
    clFinish(queue);

    struct timespec t0,t1;
    clock_gettime(CLOCK_MONOTONIC,&t0);

    int iters=0,upd;
    for(int i=0;i<V-1;i++){
        upd=0;
        clEnqueueWriteBuffer(queue,dupd,CL_TRUE,0,sizeof(int),&upd,0,NULL,NULL);
        clEnqueueNDRangeKernel(queue,kern,1,NULL,&global,&local,0,NULL,NULL);
        clEnqueueReadBuffer(queue,dupd,CL_TRUE,0,sizeof(int),&upd,0,NULL,NULL);
        iters++;
        if(!upd) break;
    }
    upd=0;
    clEnqueueWriteBuffer(queue,dupd,CL_TRUE,0,sizeof(int),&upd,0,NULL,NULL);
    clEnqueueNDRangeKernel(queue,kern,1,NULL,&global,&local,0,NULL,NULL);
    clEnqueueReadBuffer(queue,dupd,CL_TRUE,0,sizeof(int),&upd,0,NULL,NULL);
    if(upd) iters=-1;

    clFinish(queue);
    clock_gettime(CLOCK_MONOTONIC,&t1);
    *ms_out = elapsed_ms(&t0,&t1);

    clEnqueueReadBuffer(queue,ddist,CL_TRUE,0,(size_t)V*sizeof(float),hdist,0,NULL,NULL);
    clFinish(queue);

    *cpu_after_out = get_rss_kb();

    clReleaseMemObject(ds);clReleaseMemObject(dd);clReleaseMemObject(dw);
    clReleaseMemObject(ddist);clReleaseMemObject(dupd);
    return iters;
}

/* ── Small ────────────────────────────────────────────────────────────── */
static void run_small(void) {
    printf("\n╔══════════════════════════════════════════════╗\n");
    printf(  "║  SMALL GRAPH  (V=8, E=11)                   ║\n");
    printf(  "║  Hand-crafted · includes negative edge       ║\n");
    printf(  "╚══════════════════════════════════════════════╝\n");
    int V=8;
    int   s[]={0,0,1,1,2,3,4,4,0,5,6};
    int   d[]={1,2,3,4,4,5,3,5,6,7,7};
    float w[]={4,2,3,5,1,2,-2,1,8,3,10};
    int E=11;
    float *dist=(float*)malloc(V*sizeof(float));
    int   *pred=(int  *)malloc(V*sizeof(int));
    double ms, gpu_mb; long cpu_b, cpu_a;
    int iters=bf_ocl(V,E,s,d,w,0,dist,&ms,&gpu_mb,&cpu_b,&cpu_a);
    make_pred(V,E,s,d,w,dist,pred);
    printf("Source     : 0\n");
    printf("Iterations : %d  (max would be %d)\n",iters,V-1);
    printf("Time       : %.4f ms\n",ms);
    printf("Neg cycle  : %s\n",iters==-1?"YES":"no");
    print_mem(cpu_b,cpu_a,gpu_mb);
    printf("\n  %-8s  %-10s  %s\n","Vertex","Distance","Path");
    printf(    "  %-8s  %-10s  %s\n","------","--------","----");
    for(int v=0;v<V;v++){
        if(dist[v]>=INF*0.5f) printf("  %-8d  %-10s  unreachable\n",v,"inf");
        else{printf("  %-8d  %-10.0f  ",v,dist[v]);print_path(pred,0,v);printf("\n");}
    }
    free(dist);free(pred);
}

/* ── Medium ───────────────────────────────────────────────────────────── */
static void run_medium(void) {
    printf("\n╔══════════════════════════════════════════════╗\n");
    printf(  "║  MEDIUM GRAPH  (V=100,000  E=1,000,000)     ║\n");
    printf(  "║  Random · positive weights [1,50]            ║\n");
    printf(  "╚══════════════════════════════════════════════╝\n");
    int V=100000,*s,*d,E; float *w; rng=42UL;
    build(V,1000000,&s,&d,&w,&E);
    float *dist=(float*)malloc((size_t)V*sizeof(float));
    int   *pred=(int  *)malloc((size_t)V*sizeof(int));
    double ms,gpu_mb; long cpu_b,cpu_a;
    int iters=bf_ocl(V,E,s,d,w,0,dist,&ms,&gpu_mb,&cpu_b,&cpu_a);
    make_pred(V,E,s,d,w,dist,pred);
    print_stats(V,E,dist,iters,ms);
    print_mem(cpu_b,cpu_a,gpu_mb);
    printf("\nFirst 10 reachable vertices:\n");
    printf("  %-8s  %-10s  %s\n","Vertex","Distance","Path (first 8 hops)");
    printf("  %-8s  %-10s  %s\n","------","--------","-------------------");
    int shown=0;
    for(int v=1;v<V&&shown<10;v++){
        if(dist[v]<INF*0.5f){
            printf("  %-8d  %-10.0f  ",v,dist[v]);
            print_path(pred,0,v);printf("\n");shown++;
        }
    }
    free(s);free(d);free(w);free(dist);free(pred);
}

/* ── Large ────────────────────────────────────────────────────────────── */
static void run_large(void) {
    printf("\n╔══════════════════════════════════════════════╗\n");
    printf(  "║  LARGE GRAPH  (V=1,000,000  E=10,000,000)   ║\n");
    printf(  "║  Random · positive weights [1,100]           ║\n");
    printf(  "╚══════════════════════════════════════════════╝\n");
    int V=1000000,*s,*d,E; float *w; rng=99UL;
    printf("Allocating %.0f MB for edges...\n",
           10000000.0*(sizeof(int)*2+sizeof(float))/(1024*1024));
    fflush(stdout);
    build(V,10000000,&s,&d,&w,&E);
    printf("Graph built. Running Bellman-Ford...\n"); fflush(stdout);
    float *dist=(float*)malloc((size_t)V*sizeof(float));
    double ms,gpu_mb; long cpu_b,cpu_a;
    int iters=bf_ocl(V,E,s,d,w,0,dist,&ms,&gpu_mb,&cpu_b,&cpu_a);
    print_stats(V,E,dist,iters,ms);
    print_mem(cpu_b,cpu_a,gpu_mb);
    free(s);free(d);free(w);free(dist);
}

/* ── XLarge ───────────────────────────────────────────────────────────── */
static void run_xlarge(void) {
    printf("\n╔══════════════════════════════════════════════╗\n");
    printf(  "║  EXTRA-LARGE GRAPH  (V=10M  E=50M)          ║\n");
    printf(  "║  Random · positive weights [1,100]           ║\n");
    printf(  "╚══════════════════════════════════════════════╝\n");
    int V=10000000,*s,*d,E; float *w; rng=777UL;
    printf("Allocating %.0f MB for edges...\n",
           50000000.0*(sizeof(int)*2+sizeof(float))/(1024*1024));
    fflush(stdout);
    build(V,50000000,&s,&d,&w,&E);
    printf("Graph built. Running Bellman-Ford...\n"); fflush(stdout);
    float *dist=(float*)malloc((size_t)V*sizeof(float));
    if(!dist){fprintf(stderr,"malloc failed\n");return;}
    double ms,gpu_mb; long cpu_b,cpu_a;
    int iters=bf_ocl(V,E,s,d,w,0,dist,&ms,&gpu_mb,&cpu_b,&cpu_a);
    print_stats(V,E,dist,iters,ms);
    print_mem(cpu_b,cpu_a,gpu_mb);
    free(s);free(d);free(w);free(dist);
}

/* ── Negative cycle demo ──────────────────────────────────────────────── */
static void run_neg(void) {
    printf("\n╔══════════════════════════════════════════════╗\n");
    printf(  "║  NEGATIVE CYCLE DEMO                        ║\n");
    printf(  "║  Cycle 1->2->3->1  cost = 2 + (-4) + 1 = -1║\n");
    printf(  "╚══════════════════════════════════════════════╝\n");
    int V=5;
    int   s[]={0,1,2,3,0,3};
    int   d[]={1,2,3,1,4,4};
    float w[]={1,2,-4,1,5,2};
    int E=6;
    float *dist=(float*)malloc(V*sizeof(float));
    double ms,gpu_mb; long cpu_b,cpu_a;
    int iters=bf_ocl(V,E,s,d,w,0,dist,&ms,&gpu_mb,&cpu_b,&cpu_a);
    printf("Neg cycle detected : %s\n",iters==-1?"YES ✓":"no");
    printf("(distances are unreliable when a neg cycle exists)\n");
    free(dist);
}

/* ── Main ─────────────────────────────────────────────────────────────── */
int main(void) {
    printf("=======================================================\n");
    printf("  Bellman-Ford SSSP — Small / Medium / Large / XLarge \n");
    printf("=======================================================\n");
    ocl_init();
    run_small();
    run_medium();
    run_large();
    run_xlarge();
    run_neg();
    ocl_cleanup();
    printf("\n=======================================================\n");
    printf("  Done.\n");
    printf("=======================================================\n");
    return 0;
}


Overwriting sssp_opencl.c


In [ ]:
!apt-get install -y ocl-icd-opencl-dev 2>/dev/null | tail -2



ocl-icd-opencl-dev is already the newest version (2.2.14-3).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.


In [ ]:
!gcc -O3 -DCL_TARGET_OPENCL_VERSION=300 -o bf_ocl sssp_opencl.c -lOpenCL -lm && ./bf_ocl

  Bellman-Ford SSSP — Small / Medium / Large / XLarge 
  OpenCL device : NVIDIA A100-SXM4-40GB
  VRAM          : 40441 MB   CUs: 108
  Memory measured: CPU via /proc/self/status, GPU via clGetMemObjectInfo


╔══════════════════════════════════════════════╗
║  SMALL GRAPH  (V=8, E=11)                   ║
║  Hand-crafted · includes negative edge       ║
╚══════════════════════════════════════════════╝
Source     : 0
Iterations : 6  (max would be 7)
Time       : 0.4944 ms
Neg cycle  : no
Memory     : CPU=0.0 MB  GPU=0.0 MB  total=0.0 MB

  Vertex    Distance    Path
  ------    --------    ----
  0         0           0
  1         4           0 -> 1
  2         2           0 -> 2
  3         1           0 -> 2 -> 4 -> 3
  4         3           0 -> 2 -> 4
  5         3           0 -> 2 -> 4 -> 3 -> 5
  6         8           0 -> 6
  7         6           0 -> 2 -> 4 -> 3 -> 5 -> 7

╔══════════════════════════════════════════════╗
║  MEDIUM GRAPH  (V=100,000  E=1,000,000)     ║
║  Random 